In [ ]:
import os
from pathlib import Path
from tempfile import mkdtemp

DATA_PATH = Path("data") / "LN_single_spot_receiver_tensors.mat"
RESULTS_DIRECTORY = Path("results")

for variable in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "1"

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing bundled optical input: {DATA_PATH}. "
        "Extract the repository ZIP and open the notebook from the folder that contains data."
    )


In [ ]:
import csv
import hashlib
import json
import math
from contextlib import redirect_stdout
from io import StringIO
from math import factorial
from time import perf_counter
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, FancyBboxPatch, Rectangle
from mpmath import iv
from numpy.polynomial.legendre import leggauss
from scipy.integrate import quad, solve_ivp
from scipy.interpolate import CubicSpline
from scipy.io import loadmat
from scipy.linalg import eigh
from scipy.optimize import brentq, minimize, minimize_scalar
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import LinearOperator, eigsh
from scipy.special import logsumexp, ndtr


In [ ]:
def complex_gaussian(rng, shape):
    return (rng.standard_normal(shape) + 1j * rng.standard_normal(shape)) / np.sqrt(2)


def draw_matrix(rng, modes, scatterers_per_mode):
    if scatterers_per_mode == 0:
        gaussian = complex_gaussian(rng, (modes, modes))
        return (gaussian + gaussian.T) / np.sqrt(2 * modes)
    total = modes * scatterers_per_mode
    incident = complex_gaussian(rng, (total, modes))
    outgoing = complex_gaussian(rng, total)
    return incident.T @ (outgoing[:, None] * incident) / np.sqrt(total * modes)


def leading_vector(matrix, rng):
    modes = matrix.shape[0]

    def action(real_vector):
        wave = real_vector[:modes] + 1j * real_vector[modes:]
        transformed = (matrix @ wave).conj()
        return np.r_[transformed.real, transformed.imag]

    operator = LinearOperator((2 * modes, 2 * modes), matvec=action, dtype=float)
    values, vectors = eigsh(operator, k=1, which='LA', tol=1e-9,
                            v0=rng.standard_normal(2 * modes))
    vector = vectors[:modes, 0] + 1j * vectors[modes:, 0]
    assert np.linalg.norm((matrix @ vector).conj() - values[0] * vector) < 1e-7
    return vector, float(values[0])


def leading_mask(matrix, rng):
    vector, value = leading_vector(matrix, rng)
    mask = np.exp(1j * np.angle(vector)) / np.sqrt(len(vector))
    return mask, value


In [ ]:
def gaussian_wedge(s, levels):
    kappa = np.tan(np.pi / levels)
    normalizer = np.sqrt(2 / np.pi)

    def density(y):
        return np.exp(-y * y / 2)

    def arg(y):
        return np.sqrt(2) * s - y / kappa

    probability = normalizer * quad(lambda y: density(y) * ndtr(arg(y)), 0, 12,
                                    epsabs=2e-13)[0]
    derivative = normalizer / np.sqrt(np.pi) * quad(
        lambda y: density(y) * np.exp(-arg(y) ** 2 / 2), 0, 12, epsabs=2e-13)[0]
    return probability, derivative


def upper_bound():
    levels, tilt, discrete_limit = 7, 1.319, 1.7802
    m = minimize_scalar(lambda offset: offset ** 2 -
                        np.log(gaussian_wedge(tilt + offset, levels)[0]),
                        method='bounded', bounds=(0., 1.))
    s = round(tilt + m.x, 4)
    probability, derivative = gaussian_wedge(s, levels)
    slope = derivative / probability
    exponent = (np.log(levels) - tilt * discrete_limit + tilt ** 2 / 2
                + np.log(probability) + slope * (tilt - s) + slope ** 2 / 4)
    attenuation = np.sinc(1 / levels) ** 2
    assert exponent < -1e-4
    return dict(levels=levels, tilt=tilt, tangent_point=s, wedge_probability=probability,
                log_wedge_derivative=slope, tail_exponent=exponent,
                diagonal_slack_coefficient=slope, discrete_amplitude=discrete_limit,
                rounding_attenuation=attenuation, continuous_amplitude=discrete_limit / attenuation,
                continuous_enhancement_per_mode=(discrete_limit / attenuation) ** 2 / 2)


def lower_bound():
    c = np.sqrt(np.pi) / 2
    residual = 1 - c ** 2
    response, offset = .69, .3415

    def integrated(term):
        def integrand(r):
            mean_phase = np.exp(-response ** 2 * residual / (4 * (r + offset) ** 2))
            return 2 * r * np.exp(-r * r) * mean_phase * term(r)

        return quad(integrand, 0, 12, epsabs=2e-13)[0]

    a = integrated(lambda r: r)
    b = integrated(lambda r: 1 - c * r)
    d = response * residual / 2 * integrated(lambda r: 1 / (r + offset))
    field = 2 * a ** 2 + 2 * b * d / residual
    return dict(response=response, offset=offset, asymptotic_amplitude=field,
                asymptotic_enhancement_per_mode=field ** 2 / 2)


def background(matrix):
    modes = matrix.shape[0]
    return (2 * np.sum(abs(matrix) ** 2) - np.sum(abs(matrix.diagonal()) ** 2)) / modes ** 2


def optimal_values(matrix, seeds):
    modes = len(matrix)
    best_squared = -np.inf
    best_wave = None

    def objective(angles):
        wave = np.exp(1j * angles) / np.sqrt(modes)
        local = wave * (matrix @ wave)
        return -local.sum().real, 2 * local.imag

    for seed in seeds:
        angles = np.angle(seed) - .5 * np.angle(seed.T @ matrix @ seed)
        answer = minimize(objective, angles, jac=True, method='L-BFGS-B',
                          options={'maxiter': 250, 'ftol': 1e-13, 'gtol': 5e-9})
        wave = np.exp(1j * answer.x) / np.sqrt(modes)
        squared = abs(wave.T @ matrix @ wave) ** 2
        if squared > best_squared:
            best_squared, best_wave = squared, wave
    return best_squared, best_wave


def shifted_certificates(matrix, wave):
    modes = len(matrix)
    wave = wave * np.exp(-.5j * np.angle(wave.T @ matrix @ wave))
    lifted = np.block([[matrix.real, -matrix.imag], [-matrix.imag, -matrix.real]])
    diagonal_indices = np.arange(2 * modes)

    def shifted(shifts):
        a = lifted.copy()
        a[diagonal_indices, diagonal_indices] -= np.tile(shifts, 2)
        return a

    def top(shifts):
        return shifts.mean() + eigh(shifted(shifts), eigvals_only=True,
                                    subset_by_index=(2 * modes - 1, 2 * modes - 1))[0]

    shifts = modes * (wave * (matrix @ wave)).real
    raw = top(shifts)
    result = min(raw, top(np.zeros(modes)))
    for temperature in (.05, .015, .005):
        def smooth(shifts):
            values, vectors = eigh(shifted(shifts))
            scaled = values / temperature
            norm = logsumexp(scaled)
            weights = np.exp(scaled - norm)
            density = (vectors[:modes] ** 2 + vectors[modes:] ** 2) @ weights
            return shifts.mean() + temperature * norm, 1 / modes - density

        fit = minimize(smooth, shifts, jac=True, method='L-BFGS-B',
                       options={'maxiter': 85, 'gtol': 1e-7, 'ftol': 1e-11})
        shifts = fit.x
        result = min(result, top(shifts))
    return raw + 1e-10, result + 1e-10


def dense_reference_trial(modes, rng, lower, use_certificate, starts):
    matrix = draw_matrix(rng, modes, 0)
    vector, maximum = leading_vector(matrix, rng)
    gauge = np.exp(1j * np.angle(vector))
    init = gauge / np.sqrt(modes)
    signal = np.sqrt(modes) * (gauge * (matrix @ init)).imag
    amplitudes = np.sqrt(modes) * abs(vector)
    phase = -lower['response'] * signal / (amplitudes + lower['offset'])
    updated = init * np.exp(1j * phase)

    random_seeds = [np.exp(2j * np.pi * rng.random(modes)) / np.sqrt(modes)
                    for _ in range(starts)]
    best, wave = optimal_values(matrix, [init, updated, *random_seeds])
    scale = modes * background(matrix)
    row = dict(projected=float(abs(init.T @ matrix @ init) ** 2 / scale),
               analytic_update=float(abs(updated.T @ matrix @ updated) ** 2 / scale),
               feasible=float(best / scale), unconstrained=float(maximum ** 2 / scale))

    if use_certificate:
        raw, lifted = shifted_certificates(matrix, wave)
        assert np.sqrt(best) <= lifted + 2e-9
        assert lifted <= maximum + 2e-8
        row.update(local_field_upper=float(raw ** 2 / scale),
                   finite_instance_upper=float(lifted ** 2 / scale))
        if modes == 2:
            phases = np.linspace(0., 2 * np.pi, 32768, endpoint=False)
            y = np.exp(1j * phases)
            grid = abs((matrix[0, 0] + 2 * matrix[0, 1] * y + matrix[1, 1] * y ** 2) / 2)
            distance = np.pi / len(y)
            derivative_bound = abs(matrix[0, 1]) + abs(matrix[1, 1])
            assert grid.max() <= np.sqrt(best) + 1e-8
            assert np.sqrt(best) <= grid.max() + distance * derivative_bound + 1e-8
    return row


In [ ]:
iv.dps = 40


def erf_interval(x):
    if not (x.a>=0 and x.b<=iv.mpf("1.55")):
        raise ValueError(x)
    steps=28
    total=sum((-1)**k*x**(2*k+1)/(factorial(k)*(2*k+1)) for k in range(steps))
    tail=(x**(2*steps+1)/(factorial(steps)*(2*steps+1))).b
    return 2*(total+iv.mpf([-tail,tail]))/iv.sqrt(iv.pi)


def bound_report():
    k=7
    s=iv.mpf("1.5493")
    lam=iv.mpf("1.319")
    h=iv.mpf("1.7802")
    a=iv.sin(iv.pi/k)
    b=iv.cos(iv.pi/k)
    def rate(x):
        return (a/iv.sqrt(iv.pi))*iv.exp(-(a*x)**2)*(1+erf_interval(b*x))
    panels=4096
    w=s/panels
    integral=w*sum(rate(w*(i+iv.mpf(".5"))) for i in range(panels))
    error=(50*s**3/(24*panels**2)).b
    q=iv.mpf(1)/k+integral+iv.mpf([-error,error])
    derivative=rate(s)
    tilt=derivative/q
    exponent=iv.log(k)-lam*h+lam**2/2+iv.log(q)+tilt*(lam-s)+tilt**2/4
    assert exponent.b < iv.mpf("-.00068")
    return dict(wedge_probability=str(q),wedge_derivative=str(derivative),
                 exponent_at_zero_slack=str(exponent),midpoint_panels=panels,
                 midpoint_error_upper=str(error))


In [ ]:
def squared_outputs(matrix, masks):
    return abs(np.sum(masks * (matrix @ masks), axis=0)) ** 2


def split_mask(matrix):
    n = len(matrix)
    m = n // 2
    x = np.ones(n, dtype=complex) / np.sqrt(n)
    v = matrix[m:, :m] @ x[:m]
    x[m:] *= np.exp(-1j * np.angle(v))
    candidates = np.repeat(x[:, None], 4, axis=1)
    candidates[m:] *= np.array([1, 1j, -1, -1j])
    powers = squared_outputs(matrix, candidates)
    chosen = np.argmax(powers)
    assert powers[chosen] >= (2 * np.sum(abs(v)) / np.sqrt(n)) ** 2 - 1e-10
    return candidates[:, chosen], float(powers[chosen] / 2)


def one_case(a, weights, random_mask, name, rng, certify):
    k, n = a.shape
    normalization = np.sqrt(k * n)
    isolated_field = abs(weights) * np.sum(abs(a), axis=1) ** 2 / (n * normalization)
    order = np.argsort(-isolated_field)[:min(12, k)]
    row_masks = np.exp(-1j * np.angle(a[order])).T / np.sqrt(n)
    strongest = row_masks[:, 0]
    matrix = a.T @ (weights[:, None] * a) / normalization

    row_power = squared_outputs(matrix, row_masks) / 2
    largest_weight = np.argmax(abs(weights))
    weight_mask = np.exp(-1j * np.angle(a[largest_weight])) / np.sqrt(n)
    raw_brightest = abs(weight_mask.T @ matrix @ weight_mask) ** 2 / 2
    modes, singular = leading_mask(matrix, rng)
    split, split_power = split_mask(matrix)
    starts = [modes, row_masks[:, np.argmax(row_power)], split, random_mask]
    best_power, best_wave = optimal_values(matrix, starts)
    assert best_power / 2 >= np.max(row_power) - 1e-8
    assert best_power <= singular ** 2 + 1e-8
    optical_powers = abs(a @ best_wave) ** 2
    contributions = weights * (a @ best_wave) ** 2 / normalization
    peak = np.argmax(abs(contributions))
    phase_capacity = np.sum(abs(a[peak])) ** 2 / n
    raw_endpoint = float(np.max(abs(weights)) ** 2 * n / k)
    result = dict(
        distribution=name, endpoints=raw_endpoint, best=float(best_power / 2),
        strongest_one=float(row_power[0]), strongest_weight=float(raw_brightest),
        shortlist=float(row_power.max()), split=float(split_power),
        takagi=float(abs(modes.T @ matrix @ modes) ** 2 / 2),
        unrestrained=float(singular ** 2 / 2),
        dominant_term_to_field=float(abs(contributions[peak]) / abs(contributions.sum())),
        largest_site_match=float(optical_powers[peak] / phase_capacity),
        strongest_one_fraction=float(2 * row_power[0] / best_power),
        shortlist_fraction=float(2 * row_power.max() / best_power),
        strongest_mask_power=float(abs(strongest.T @ matrix @ strongest) ** 2 / 2),
        isolated_peak=float(isolated_field[order[0]] ** 2 / 2),
    )
    if certify:
        _, upper = shifted_certificates(matrix, best_wave)
        assert upper ** 2 >= best_power - 1e-8
        result['finite_upper'] = float(upper ** 2 / 2)
    return result


def crossover_trial(n, k, rng, certify):
    a = complex_gaussian(rng, (k, n))
    gaussian = complex_gaussian(rng, k)
    radii = abs(gaussian)
    phases = gaussian / radii
    cap = 1.5
    clipped = phases * np.minimum(radii, cap) / np.sqrt(-np.expm1(-cap ** 2))
    initial = np.exp(2j * np.pi * rng.random(n)) / np.sqrt(n)
    variants = [('gaussian', gaussian), ('clipped_1p5', clipped), ('unit_magnitude', phases)]
    solver_seed = int(rng.integers(2 ** 32))
    return [
        one_case(a, values, initial, name, np.random.default_rng(solver_seed), certify)
        for name, values in variants
    ]


In [ ]:
def sparse_scatterers(rng):
    rows = []
    count = 2048
    for n in (64, 256, 1024, 4096):
        for ratio in (.5, 2, 8):
            k = int(n * ratio)
            values = []
            for first in range(0, count, 64):
                outgoing = complex_gaussian(rng, (64, k))
                selected = np.argmax(abs(outgoing), axis=1)
                first_index = np.arange(64)
                strongest = abs(outgoing[first_index, selected])
                outgoing[first_index, selected] = 0.
                row_sum = abs(complex_gaussian(rng, (64, n))).sum(axis=1) / np.sqrt(n)
                other_fields = complex_gaussian(rng, (64, k))
                cross = (outgoing * other_fields ** 2).sum(axis=1)
                power = abs(strongest * row_sum ** 2 + cross) ** 2 / (2 * k * n)
                values.extend(power.tolist())
            a = np.asarray(values)
            rows.append(dict(modes=n, scatterers=k, ratio=ratio, draws=count,
                             mean_enhancement_per_mode=float(a.mean()),
                             above_gaussian_bound=float(np.mean(a > 1.813934)),
                             leading_log_term=float(np.pi ** 2 * np.log(k) / (32 * ratio))))
            print('sparse', rows[-1], flush=True)
    return rows


In [ ]:
def finite_size_mean(n,k):
    h=math.fsum(1/j for j in range(1,k+1))
    m1=math.sqrt(math.pi)/2
    m3=3*math.sqrt(math.pi)/4
    fourth=(2*n+4*n*(n-1)*m1*m3+3*n*(n-1)
            +6*n*(n-1)*(n-2)*m1*m1+n*(n-1)*(n-2)*(n-3)*m1**4)/n**2
    return (h*fourth+2*(k-h))/(2*k*n)


In [ ]:
CHI2 = 7 / 12


def polar_tall(gaussian):
    values, vectors = eigh(gaussian.conj().T @ gaussian)
    inverse = (vectors / np.sqrt(values)) @ vectors.conj().T
    return gaussian @ inverse


def make_propagation(n, k, rng, extra_ports):
    d = int(np.ceil(extra_ports * max(n, k)))
    if d == n and k < n:
        a = polar_tall(complex_gaussian(rng, (n, k))).conj().T
    else:
        a = polar_tall(complex_gaussian(rng, (d, n)))[:k]
    assert np.linalg.norm(a, 2) <= 1 + 2e-9
    return a, d


def mask_for_outputs(a, b, selected):
    desired = np.exp(.5j * np.angle(b[selected])) / np.sqrt(len(selected))
    launch = a[selected].conj().T @ desired
    return np.exp(1j * np.angle(launch)) / np.sqrt(a.shape[1])


def weak_test(a, d, b, chi, seed, rng):
    n = a.shape[1]
    coefficient = b.conj() * chi
    capacities = abs(coefficient) * np.sum(abs(a), axis=1) ** 2 / n
    order = np.argsort(-capacities)[:min(8, len(b))]
    masks = [mask_for_outputs(a, b, order[i:i+1]) for i in range(len(order))]
    scale = np.sqrt(n * CHI2) / d
    matrix = a.T @ (coefficient[:, None] * a) / scale
    scores = [abs(x.T @ matrix @ x) ** 2 for x in masks]
    projected, upper = leading_mask(matrix, rng)
    best, wave = optimal_values(matrix, [masks[int(np.argmax(scores))], projected, seed])
    best *= scale ** 2
    row = scores[0] * scale ** 2
    upper_power = (upper * scale) ** 2
    passive_bound = np.max(abs(coefficient)) ** 2
    assert best + 1e-14 >= row
    assert best <= upper_power + 1e-13
    assert upper_power <= passive_bound + 1e-12
    incoming = a @ wave
    baseline = 2 * CHI2 / (d * (d + 1))
    return dict(single_over_best=float(row / best), best_per_input=float(best / (baseline * n)),
                best_absolute_coeff=float(best), single_absolute_coeff=float(row),
                physical_upper=float(passive_bound), incident_power=float(np.vdot(incoming, incoming).real),
                max_local_pump=float(max(abs(incoming) ** 2)),
                largest_weight=float(np.max(abs(coefficient))))


def draw_target(k, rng):
    z = complex_gaussian(rng, k)
    speckled = z / np.linalg.norm(z)
    z /= abs(z)
    flat = z / np.sqrt(k)
    return speckled, flat


def summarize_passive_trials(rows):
    stats = {}
    for key in rows[0]:
        values = np.array([row[key] for row in rows])
        stats[key] = dict(mean=float(values.mean()), se=float(values.std(ddof=1) / np.sqrt(len(values))))
    return stats


def conversion(a, b, chi, drive, wave, need_gradient=False):
    pump = a @ wave
    radius = abs(pump)
    safe = np.maximum(radius, 1e-15)
    gchi = drive * chi
    local = np.tanh(gchi * radius)
    h = local / safe
    h = np.where(radius > 1e-15, h, gchi)
    signal = pump ** 2 * h
    field = np.vdot(b, signal)
    incident_power = float(np.vdot(pump, pump).real)
    converted = float(np.vdot(signal, signal).real)
    remaining = float(np.sum(radius ** 2 * (1 - local ** 2)))
    assert abs(remaining + converted - incident_power) < 1e-10
    assert abs(field) ** 2 <= converted + 1e-12 and incident_power <= 1 + 1e-9
    if not need_gradient:
        return abs(field) ** 2, converted, incident_power
    dh_over_r = (gchi * radius * (1 - local ** 2) - local) / safe ** 3
    small = gchi * radius < 1e-4
    dh_over_r[small] = -2 * gchi[small] ** 3 / 3
    bstar = b.conj()
    nonlinear = bstar * pump ** 2 * dh_over_r / 2
    positive = (a.T @ (bstar * 2 * pump * h + nonlinear * pump.conj())) * (1j * wave)
    negative = (a.conj().T @ (nonlinear * pump)) * (-1j * wave.conj())
    return abs(field) ** 2, 2 * (field.conj() * (positive + negative)).real


def saturating_best(a, b, chi, drive, seeds):
    k, n = a.shape
    objective_scale = max(drive ** 2 * n / k ** 2, 1e-8)

    def objective(angles):
        current = np.exp(1j * angles) / np.sqrt(n)
        power, gradient = conversion(a, b, chi, drive, current, True)
        return -power / objective_scale, -gradient / objective_scale

    f = np.angle(seeds[0])
    _, derivative = objective(f)
    h = 2e-5
    check = f.copy()
    check[0] += h
    check2 = f.copy()
    check2[0] -= h
    centered = (objective(check)[0] - objective(check2)[0]) / (2 * h)
    assert abs(derivative[0] - centered) < 1e-7
    best = (0., 0., 0.)
    for seed in seeds:
        start = np.angle(seed)
        found = minimize(objective, start, jac=True, method='L-BFGS-B',
                         options={'maxiter': 220, 'gtol': 2e-8, 'ftol': 1e-12})
        wave = np.exp(1j * found.x) / np.sqrt(n)
        result = conversion(a, b, chi, drive, wave)
        if result[0] > best[0]:
            best = result
    return best


In [ ]:
def sample_pumps(n, k, d, counts, chi, b, rng):
    highest = np.argsort(-abs(b * chi))
    h = max(counts)
    selected, remaining = highest[:h], highest[h:]
    full_selected = polar_tall(complex_gaussian(rng, (d, h))).conj().T
    selected_rows = full_selected[:, :n]
    masks = np.stack([
        mask_for_outputs(selected_rows, b[selected], np.arange(count))
        for count in counts
    ], axis=1)
    observed = selected_rows @ masks
    gram = masks.conj().T @ masks - observed.conj().T @ observed
    eigenvalues, eigenvectors = eigh(gram)
    assert min(eigenvalues) >= -1e-10
    square_root = (eigenvectors * np.sqrt(np.maximum(eigenvalues, 0))) @ eigenvectors.conj().T
    completion = polar_tall(complex_gaussian(rng, (d - h, len(counts)))) @ square_root
    assert np.max(abs(gram - completion.conj().T @ completion)) < 1e-9
    fields = np.empty((k, len(counts)), dtype=complex)
    fields[selected] = observed
    fields[remaining] = completion[:k-h]
    assert np.max(np.sum(abs(fields) ** 2, axis=0)) <= 1 + 1e-9
    return fields


In [ ]:
def dense_fresh_trial(seed, n):

    rng = np.random.default_rng(seed)
    tensor = draw_matrix(rng, n, 0)
    r, norm = leading_vector(tensor, rng)
    gauge = np.exp(1j * np.angle(r))
    projected = gauge / np.sqrt(n)
    params = lower_bound()
    signal = np.sqrt(n) * (gauge * (tensor @ projected)).imag
    change = -params["response"] * signal / (np.sqrt(n) * abs(r) + params["offset"])
    corrected = projected * np.exp(1j * change)
    random_masks = np.exp(2j*np.pi*rng.random((n, 3072))) / np.sqrt(n)
    powers = abs(np.einsum("ij,ij->j", random_masks, tensor @ random_masks)) ** 2
    analytic_background = background(tensor)
    phase_grid = np.exp(2j*np.pi*np.arange(3) / 3)
    tiny = tensor[:5, :5]
    mesh = np.indices((3,)*5).reshape(5, -1)
    exact = phase_grid[mesh] / np.sqrt(5)
    np.testing.assert_allclose(np.mean(abs(np.einsum("ij,ij->j", exact, tiny @ exact)) ** 2),
                               background(tiny), rtol=2e-14)
    baseline = abs(projected.T @ tensor @ projected)**2
    new_power = abs(corrected.T @ tensor @ corrected)**2
    best, _ = optimal_values(tensor, [projected, corrected, *random_masks[:, :4].T])
    return dict(seed=seed, n=n, background=analytic_background, background_mc=float(powers.mean()),
                projected=float(baseline), corrected=float(new_power), best=best, amplitude_bound=float(norm))


def passive_group_trial(seed, n, extra):

    rng = np.random.default_rng(seed)
    k, d = n, int(n * extra)
    counts = [1, 2, 4, 8, 16, 32, 64, 128, 256]
    chi = rng.uniform(.5, 1., k)
    b, _ = draw_target(k, rng)
    u = sample_pumps(n, k, d, counts, chi, b, rng)
    radius = abs(u)
    fields = {}
    for g in (.15, 2., 5., 8., 12.):
        conversion = u*u*np.tanh(g*chi[:, None]*radius) / np.maximum(radius, 1e-15)
        power = abs(b.conj() @ conversion)**2
        assert np.all(power <= np.sum(abs(conversion)**2, axis=0)+1e-10)
        fields[str(g)] = power.tolist()
    return dict(seed=seed, n=n, k=k, d=d, counts=counts, power=fields)


def theory_cases(mode):
    if mode == "dense":
        return [(76543+1009*i+n, n) for n, reps in ((128, 20), (512, 16), (1024, 12)) for i in range(reps)]
    if mode == "crossover":
        return [(91801+1021*i+n, n, s) for n in (128, 256, 512)
                for s in (.15, .6, 2., 8.) for i in range(10)]
    return [(81421+1061*i+n, n, extra) for n in (2048, 8192)
            for extra in (1., 1.5) for i in range(24)]


In [ ]:
def screened_passive_trial(seed):

    n=8192
    g=8.
    counts=(1,2,4,8,16,32,64,128,256)
    rng=np.random.default_rng(seed)
    chi=rng.uniform(.5,1.,n)
    b,_=draw_target(n,rng)
    order=np.argsort(-abs(b*chi))
    raw_order=np.argsort(-abs(b))
    selected=np.r_[order[:256],raw_order[:32][~np.isin(raw_order[:32],order[:256])]]
    h=len(selected)
    remaining=np.flatnonzero(~np.isin(np.arange(n),selected))
    a=polar_tall(complex_gaussian(rng,(n,h))).conj().T
    row_powers=np.sum(abs(a),axis=1)**2/n
    isolated=abs(b[selected])*np.sqrt(row_powers)*np.tanh(g*chi[selected]*np.sqrt(row_powers))
    strongest=np.argsort(-isolated)[:32]
    targets=np.r_[0,strongest[strongest!=0]]
    single=np.exp(-1j*np.angle(a[targets])).T/np.sqrt(n)
    grouped=np.column_stack([mask_for_outputs(a,b[selected],np.arange(c)) for c in counts])
    x=np.column_stack((single,grouped))
    u=a@x
    gram=x.conj().T@x-u.conj().T@u
    w,v=np.linalg.eigh(gram)
    assert w.min()>-1e-10
    root=(v*np.sqrt(np.maximum(w,0)))@v.conj().T
    complement=polar_tall(complex_gaussian(rng,(n-h,x.shape[1])))@root
    assert np.max(abs(complement.conj().T@complement-gram))<1e-9
    result=np.empty((n,x.shape[1]),dtype=complex)
    result[selected]=u
    result[remaining]=complement
    radius=abs(result)
    emitted=result**2*np.tanh(g*chi[:,None]*radius)/np.maximum(radius,1e-15)
    powers=abs(b.conj()@emitted)**2
    assert np.max(np.sum(abs(result)**2,axis=0))<1+1e-9
    assert np.all(powers<=np.sum(abs(emitted)**2,axis=0)+1e-10)
    return dict(seed=seed,single_candidates=len(targets),dominant_weight=float(powers[0]),
                screened=float(powers[:len(targets)].max()),groups=powers[len(targets):].tolist(),
                tested_group_sizes=counts)


In [ ]:
def _rk4_step(pump, sh, rhs, h):
    a, b = rhs(pump, sh)
    c, d = rhs(pump + h * a / 2, sh + h * b / 2)
    e, f = rhs(pump + h * c / 2, sh + h * d / 2)
    i, j = rhs(pump + h * e, sh + h * f)
    return pump + h * (a + 2*c + 2*e + i) / 6, sh + h * (b + 2*d + 2*f + j) / 6


def radial_table(g, chi, delta, loss, intervals=256, steps=160):
    p = np.linspace(0, 1, intervals + 1)[:, None]
    k2 = (g * chi) ** 2 * p
    u = np.ones(k2.shape, dtype=complex)
    h = np.zeros_like(u)
    low, high = loss

    def rhs(a, b):
        return -k2 * a.conj() * b - low * a / 2, a*a - (high/2 + 1j*delta) * b

    for _ in range(steps):
        u, h = _rk4_step(u, h, rhs, 1 / steps)
    surviving = abs(u) ** 2 + k2 * abs(h) ** 2
    if max(loss) == 0:
        assert np.max(abs(surviving - 1)) < 3e-6
    else:
        assert np.max(surviving) <= 1 + 3e-7
    if np.all(delta == 0) and max(loss) == 0:
        t = np.sqrt(k2)
        exact = np.ones_like(t)
        np.divide(np.tanh(t), t, out=exact, where=t != 0)
        assert np.max(abs(h - exact)) < 3e-7
    return CubicSpline(p[:, 0], g * chi * h, axis=0).c


def radial_values(grid, pump_power):
    count = grid.shape[1]
    site = np.arange(len(pump_power))
    index = np.minimum((count * np.maximum(pump_power, 0)).astype(int), count - 1)
    p = pump_power - index / count
    cubic, square, linear, constant = grid[:, index, site]
    return ((cubic*p + square)*p + linear)*p + constant, (3*cubic*p + 2*square)*p + linear


def weak_strength(chi, delta, loss):
    q = loss[1]/2 + 1j*delta
    k = q - loss[0]
    val = np.ones_like(k)
    np.divide(np.expm1(k), k, out=val, where=abs(k) > 1e-12)
    return chi * np.exp(-q) * val


def field(a, b, grid, x, grad=False):
    u = a @ x
    f, fp = radial_values(grid, abs(u)**2)
    bstar = b.conj()
    q = (bstar * f) @ (u*u)
    if not grad:
        return float(abs(q)**2)
    z1 = bstar * (2*u*f + u*u*u.conj()*fp)
    z2 = bstar * u**3 * fp
    dq = 1j*x*(a.T @ z1) - 1j*x.conj()*(a.conj().T @ z2)
    return float(abs(q)**2), 2*(q.conj() * dq).real


def masks(a, signal):
    n = a.shape[1]
    strength = abs(signal) * np.sum(abs(a), axis=1)**2/n
    order = np.argsort(-strength)
    def mask(ids):
        desired = np.exp(-.5j * np.angle(signal[ids])) / np.sqrt(len(ids))
        return np.exp(1j * np.angle(a[ids].conj().T @ desired)) / np.sqrt(n)
    singles = [mask(order[i:i+1]) for i in range(min(8, len(order)))]
    counts = [c for c in (1, 2, 4, 8, 16, 32) if c <= len(a)]
    groups = [mask(order[:c]) for c in counts]
    return singles, groups, counts, order


def weak_design(a, b, chi, delta, loss, rng):
    coeff = b.conj() * weak_strength(chi, delta, loss)
    t = a.T @ (coeff[:, None] * a)
    scale = max(np.linalg.norm(t), 1e-14)
    t = t / scale
    start, upper = leading_mask(t, rng)
    singles, groups, counts, order = masks(a, coeff)
    random = np.exp(2j*np.pi*rng.random(a.shape[1])) / np.sqrt(a.shape[1])
    init = [start, groups[0], random]
    power, wave = optimal_values(t, init)
    return singles, groups, counts, order, wave, power*scale**2, (upper*scale)**2


def strong_design(a, b, table, initial, scale):
    scale = max(scale, 1e-12)
    def objective(phase):
        x = np.exp(1j*phase) / np.sqrt(len(phase))
        q, d = field(a, b, table, x, True)
        return -q/scale, -d/scale

    best = 0.
    best_mask = initial[0]
    for x in initial:
        candidate = field(a, b, table, x)
        if candidate > best:
            best, best_mask = candidate, x
        result = minimize(objective, np.angle(x), jac=True, method='L-BFGS-B',
                          options={'gtol': 3e-8, 'ftol': 1e-12, 'maxiter': 190})
        wave = np.exp(1j*result.x) / np.sqrt(len(x))
        power = field(a, b, table, wave)
        if power > best:
            best, best_mask = power, wave
    return best, best_mask


In [ ]:
PHYSICAL_WIDTH = 108.


STRESSES = {
    'matched': (0., 0., (0., 0.)),
    'nominal': (.55, .24, (.08, .16)),
    'detuned': (2.5, .7, (.20, .35)),
    'severe': (5.8, 1.15, (.38, .60)),
    'loss_only': (.55, .24, (.38, .60)),
    'mismatch_only': (5.8, 1.15, (.08, .16)),
}


def diffraction_geometry(seed, side, controls=6):
    rng = np.random.default_rng(seed)
    s = (np.arange(side) + .5) * PHYSICAL_WIDTH / side - PHYSICAL_WIDTH/2
    col, row = np.meshgrid(s, s, indexing='ij')
    sites = np.c_[col.ravel(), row.ravel()] + rng.uniform(-.65, .65, (side**2, 2))
    plates = []
    for _ in range(2):
        k = rng.integers(-8, 9, (26, 2))
        zero = np.sum(k*k, axis=1)==0
        k[zero, 0] = 1
        weight = rng.normal(size=len(k))/np.maximum(np.sum(k*k, axis=1), 2)**.38
        phase = rng.uniform(0, 2*np.pi, len(k))
        plates.append((k, phase, weight/np.linalg.norm(weight)))
    return dict(sites=sites, chi=rng.uniform(.5, 1., side**2), detuning=rng.standard_normal(side**2),
                plates=plates, controls=controls, pitch=PHYSICAL_WIDTH/side)


def spatial_modes(x, y, positions, waist, support):
    rows, cols, data = [], [], []
    for index, (a, b) in enumerate(positions):
        radius2 = (x-a)**2+(y-b)**2
        select = np.flatnonzero(radius2.ravel() < support**2)
        assert len(select) > 0
        weight = np.exp(-radius2.ravel()[select]/waist**2)
        rows.extend(select.tolist())
        cols.extend([index]*len(select))
        data.extend((weight/np.linalg.norm(weight)).tolist())
    assert len(set(rows)) == len(rows)
    return csc_matrix((data,(rows,cols)), shape=(x.size,len(positions)))


def diffraction_maps(config, step=3., width=288., screen_scale=1., output_waist=11., verify=False, detector_offset=(0.,0.), ports=None):
    count = int(round(width/step))
    axis = (np.arange(count)-count//2+.5)*step
    x,y=np.meshgrid(axis,axis,indexing='ij')
    apertures = np.exp(-((x/105)**12+(y/105)**12))
    kx,ky=np.meshgrid(np.fft.fftfreq(count,step),np.fft.fftfreq(count,step),indexing='ij')
    rho2=kx*kx+ky*ky
    heights=[]
    for k, phi, weights in config['plates']:
        height = np.zeros_like(x)
        for (a,b), p, w in zip(k,phi,weights):
            height += w*np.cos(2*np.pi*(a*x+b*y)/192+p)
        heights.append(1.15 * screen_scale * height)

    def make_ops(wavelength):
        shifts=[]
        kz=(2*np.pi/wavelength)*(np.sqrt(1 - wavelength**2 * rho2 + 0j)-1)
        n = 1.4496 if wavelength > .8 else 1.4607
        for z in (563., 686., 741.):
            shifts.append(np.exp(1j*z*kz)[:,:,None])
        phases=[(apertures*np.exp(2j*np.pi*(n-1)*h/wavelength))[:,:,None] for h in heights]

        def advance(u, m):
            return np.fft.ifft2(np.fft.fft2(u,axes=(0,1),norm='ortho')*m,
                               axes=(0,1),norm='ortho')
        def forward(u):
            return advance(phases[1]*advance(phases[0]*advance(u,shifts[0]),shifts[1]),shifts[2])
        def reverse(u):
            return advance(phases[0]*advance(phases[1]*advance(u,shifts[2]),shifts[1]),shifts[0])
        return forward, reverse

    f, _ = make_ops(1.064)
    fh, rh=make_ops(.532)
    blocks = config['controls']
    macro = np.linspace(-PHYSICAL_WIDTH/2, PHYSICAL_WIDTH/2, blocks+1)
    inputs = []
    for i in range(blocks):
        for j in range(blocks):
            cell = ((x>=macro[i])&(x<macro[i+1])&(y>=macro[j])&(y<macro[j+1]))
            inputs.append(cell / np.sqrt(np.sum(cell)))
    pumped=f(np.stack(inputs,axis=2)).reshape(x.size,len(inputs))
    s=config['sites']
    radius = min(5.6,.4*config['pitch'])
    project_p=spatial_modes(x,y,s,3.7,radius)
    project_h=spatial_modes(x,y,s,2.9,radius)
    a=project_p.T @ pumped
    if ports is None:
        detector=np.exp(-((x-9-detector_offset[0])**2+(y+6-detector_offset[1])**2)/output_waist**2)
        detector /= np.linalg.norm(detector)
    else:
        detector = np.stack([np.exp(-((x-xx)**2+(y-yy)**2)/output_waist**2) *
                             (((x-xx)**2+(y-yy)**2)<13**2) for xx,yy in ports],axis=2)
        detector /= np.linalg.norm(detector,axis=(0,1))
    if ports is None:
        b=(project_h.T @ fh(detector[:,:,None]).reshape(-1)).conj()
    else:
        b=(project_h.T @ fh(detector).reshape(x.size,len(ports))).conj()
    assert np.linalg.norm(a,2) <= 1+1e-10 and np.linalg.norm(b) <= 1+1e-10
    if verify and ports is None:
        coeff=complex_gaussian(np.random.default_rng(102),len(s))
        probe=(project_h @ coeff).reshape(count,count,1)
        q=np.vdot(detector,rh(probe)[:,:,0])
        assert abs(q-np.vdot(b,coeff)) < 1e-10
    return a,b


In [ ]:
def matrix_evolution(h, length, alpha=0.):
    values, vectors = eigh(h)
    return (vectors * np.exp((-1j*values-alpha/2)*length)) @ vectors.T


def coupled_model(seed, side, active, delta, loss):
    rng=np.random.default_rng(seed)
    coords=np.array([(i,j) for i in range(side) for j in range(side)])
    d=np.sum(abs(coords[:,None,:]-coords[None,:,:]),axis=-1)
    adjacent=d==1
    size=len(coords)
    perturb=rng.normal(0,.12,(size,size))
    perturb=(perturb+perturb.T)/2
    geometry=rng.normal(size=size)
    h1=adjacent*np.exp(perturb)+np.diag(.36*geometry)
    h2=.46*adjacent*np.exp(1.6*perturb)+np.diag(.65*geometry+.16*rng.normal(size=size))
    location=(side//2)*side+side//2
    target=np.eye(size)[:, location]
    a=matrix_evolution(h1,2.1,.045)
    b=matrix_evolution(h2,2.55,.085).conj().T @ target
    chi=rng.uniform(.5,1.,size)
    return dict(a=a,b=b, h1=active*h1,h2=active*h2+delta*np.eye(size),
                chi=chi, loss=loss)


def born(m, panels):
    x,w=leggauss(panels)
    matrix=np.zeros_like(m['a'])
    for t, weight in zip((1+x)/2,w/2):
        a=matrix_evolution(m['h1'],t,m['loss'][0]) @ m['a']
        b=matrix_evolution(m['h2'],1-t,m['loss'][1]).conj().T @ m['b']
        matrix += (-1j*weight) * a.T @ ((b.conj()*m['chi'])[:,None]*a)
    return matrix


def propagator(m, x, g, steps, gradient=False):
    k=g*m['chi']
    lin1=-1j*m['h1']-m['loss'][0]/2*np.eye(len(k))
    lin2=-1j*m['h2']-m['loss'][1]/2*np.eye(len(k))
    b=m['b']
    pump=m['a']@x
    pair=np.stack((pump,np.zeros_like(pump)))
    if x.ndim>1:
        k=k[:,None]
    initial=np.sum(abs(pair)**2,axis=(0,1))
    h=1/steps
    hist=[]

    def rhs(y):
        u,v=y
        return np.array([lin1@u-1j*k*u.conj()*v,lin2@v-1j*k*u*u])

    for _ in range(steps):
        q1=rhs(pair)
        stage2=pair+h*q1/2
        q2=rhs(stage2)
        stage3=pair+h*q2/2
        q3=rhs(stage3)
        stage4=pair+h*q3
        q4=rhs(stage4)
        if gradient:
            hist.append((pair,stage2,stage3,stage4))
        pair=pair+h*(q1+2*q2+2*q3+q4)/6
    energy=np.sum(abs(pair)**2,axis=(0,1))
    assert np.max(energy-initial) < 2e-5
    if max(m['loss']) == 0:
        assert np.max(abs(energy-initial)) < 2e-5
    field=b.conj()@pair[1]
    power=abs(field)**2
    assert np.all(power <= energy+1e-10)
    if not gradient:
        return power,field,energy

    p=np.array([np.zeros(len(k),complex),2*b*field])
    l1=lin1.conj().T
    l2=lin2.conj().T
    def transpose(state, co):
        a, c=state
        pu,pv=co
        return np.array([l1@pu-1j*k*c*pu.conj()+2j*k*a.conj()*pv,
                         l2@pv+1j*k*a*pu])
    for s1,s2,s3,s4 in reversed(hist):
        r4=transpose(s4,h*p/6)
        r3=transpose(s3,h*p/3+h*r4)
        r2=transpose(s2,h*p/3+h*r3/2)
        r1=transpose(s1,h*p/6+h*r2/2)
        p=p+r1+r2+r3+r4
    adj=m['a'].conj().T @ p[0]
    gphase=(adj.conj()*1j*x).real
    return power,gphase,energy


def optimize_coupled_phase(m,g,start,scale,steps):
    def func(phase):
        x=np.exp(1j*phase)/np.sqrt(len(phase))
        p,d,_=propagator(m,x,g,steps,True)
        return -p/scale,-d/scale
    angles=np.angle(start)
    f0, d0=func(angles)
    direction=np.random.default_rng(7831).normal(size=len(angles))
    direction/=np.linalg.norm(direction)
    eps=1e-5
    observed=(func(angles+eps*direction)[0]-func(angles-eps*direction)[0])/(2*eps)
    assert abs(observed-d0@direction) < 4e-6
    path=minimize(func,angles,jac=True,method='L-BFGS-B',
                  options={'maxiter':65,'gtol':8e-7,'ftol':2e-11})
    x=np.exp(1j*path.x)/np.sqrt(len(angles))
    return float(propagator(m,x,g,steps*2)[0])


In [ ]:
def model_masks(a, weights, weak_tensor, seed):

    n = a.shape[1]
    rng = np.random.default_rng(seed + 126711)
    _, groups, counts, order = masks(a, weights)
    all_single = np.exp(1j * np.angle(a[order].conj().T)) / np.sqrt(n)
    scaled = weak_tensor / np.linalg.norm(weak_tensor)
    projected, _ = leading_mask(scaled, rng)
    random = np.exp(2j * np.pi * rng.random((n, 12))) / np.sqrt(n)
    single_fields = abs(np.einsum('ij,ij->j', all_single, scaled @ all_single)) ** 2
    best_single = single_fields.argsort()[-4:]
    starts = [projected, *groups, *all_single[:, best_single].T, *random[:, :8].T]
    _, weak_mask = optimal_values(scaled, starts)
    return all_single, groups, counts, weak_mask, projected, random


def diffraction_trial(seed, optimize):

    config = diffraction_geometry(seed, 8, controls=8)
    a, b = diffraction_maps(config, step=1.125, verify=True)
    center, spread, loss = STRESSES["nominal"]
    chi = config["chi"]
    delta = center + spread * config["detuning"]
    signal = b.conj() * weak_strength(chi, delta, loss)
    t = a.T @ (signal[:, None] * a)
    single, groups, counts, weak, projected, random = model_masks(a, signal, t, seed)
    old_weak = weak_design(a, b, chi, delta, loss, np.random.default_rng(seed + 835))[4]
    out = []
    for g in (0.2, 8., 16.):
        table = radial_table(g, chi, delta, loss)
        evaluate = lambda x: field(a, b, table, x)
        s = np.array([evaluate(x) for x in single.T])
        gp = np.array([evaluate(x) for x in groups])
        weak_p, old_weak_p = map(evaluate, [weak, old_weak])
        ibest = int(s.argmax())
        ibest_group = int(gp.argmax())
        power = None
        if optimize and g == 16:
            starts = [single[:, ibest], groups[ibest_group], weak, projected, old_weak, *random[:, :2].T]
            power, _ = strong_design(a, b, table, starts, max(s.max(), gp.max(), weak_p))
        out.append(dict(drive=g, single=float(s.max()), single_8=float(s[:8].max()),
                        best_single_rank=ibest + 1, groups=gp.tolist(), counts=counts,
                        weak=weak_p, old_weak=old_weak_p, optimized=power))
    return out


def waveguide_trial(seed, active, optimize, mismatch=.65, loss=(.08,.13), physical_sh_coupling=None):

    m = coupled_model(seed, 6, active, mismatch, loss)
    if physical_sh_coupling is not None:
        h2 = m["h2"]
        diag = np.diag(np.diag(h2))
        m["h2"] = diag + physical_sh_coupling * (h2 - diag)
    a = matrix_evolution(m['h1'], .5, m['loss'][0]) @ m['a']
    b = matrix_evolution(m['h2'], .5, m['loss'][1]).conj().T @ m['b']
    t = born(m, 24)
    signal = b.conj() * m['chi']
    single, groups, counts, weak, projected, random = model_masks(a, signal, t, seed)
    rng = np.random.default_rng(seed + 831)
    old_proj, _ = leading_mask(t / np.linalg.norm(t), rng)
    n = len(t)
    old_random = np.exp(2j*np.pi*rng.random((n, 16)))[:, 0] / np.sqrt(n)
    old_weak = optimal_values(t / np.linalg.norm(t), [old_proj, groups[0], old_random])[1]
    weights = np.column_stack((single, *groups, weak, old_weak, projected, random[:, :3]))
    out = []
    for g in (.15, 2., 4., 6., 8., 12., 16.):
        steps = 160 if g == 16 else 112
        direct, _, _ = propagator(m, weights, g, steps * 2)
        s = direct[:n]
        gp = direct[n:n+len(groups)]
        weak_p, old_weak_p = direct[n+len(groups):n+len(groups)+2]
        refined = None
        if optimize and g == 8:
            starts = [int(s.argmax()), n + int(gp.argmax()), n + len(groups), n + len(groups) + 1, len(direct)-1]
            refined = float(max(direct.max(), *(optimize_coupled_phase(m, g, weights[:, i], direct.max(), steps) for i in dict.fromkeys(starts))))
        weak_intensities = g*g*abs(np.einsum('ij,ij->j', weights, t @ weights))**2
        out.append(dict(drive=g, single=float(s.max()), single_8=float(s[:8].max()),
                        best_single_rank=int(s.argmax())+1, groups=gp.tolist(), counts=counts,
                        weak=float(weak_p), old_weak=float(old_weak_p), optimized=refined,
                        born_group=weak_intensities[n:n+len(groups)].tolist(),
                        born_weak=float(weak_intensities[n+len(groups)])))
    return out


def optical_task(spec):
    model, cohort, seed, active, optimize = spec
    if model == "diffraction":
        result = diffraction_trial(seed, optimize)
    elif model == "mapped":
        result = waveguide_trial(seed, active, optimize, physical_sh_coupling=0.)
    else:
        result = waveguide_trial(seed, active, optimize)
    return dict(model=model, cohort=cohort, seed=seed, active=active, results=result)


def optical_cases(mode):
    if mode == "diffraction":
        return [(mode, label, seed, None, True)
                for label, seeds in (("archive", [25712+901*i for i in range(9)]),
                                     ("new", [43117+1013*i for i in range(9)])) for seed in seeds]
    if mode == "mapped":
        return [(mode, "new", 65379+1021*i, .45, False) for i in range(16)]
    return [(mode, label, seed, active, label == "new" and active == .45)
            for active in (.45, 1.6)
            for label, seeds in (("archive", [17007+919*i for i in range(12)]),
                                 ("new", [65379+1021*i for i in range(16)]))
            for seed in seeds]


In [ ]:
def coupling_trial(task):

    seed, hopping, disorder = task
    mismatch = .65
    drive = 8.
    m = coupled_model(seed, 6, hopping, mismatch, (.08,.13))
    for name, center in (('h1',0.), ('h2',mismatch)):
        h = m[name]
        diagonal = np.diag(h).copy()
        h[np.diag_indices_from(h)] = center+(diagonal-center)*disorder/hopping
    a = matrix_evolution(m['h1'], .5, m['loss'][0]) @ m['a']
    b = matrix_evolution(m['h2'], .5, m['loss'][1]).conj().T @ m['b']
    t = born(m, 24)
    one, groups, counts, weak, _, _ = model_masks(a, b.conj()*m['chi'], t, seed)
    all_masks = np.column_stack((one, *groups, weak))
    powers = propagator(m, all_masks, drive, 224)[0]
    n = a.shape[1]
    return dict(seed=seed,hopping=hopping,disorder=disorder,
                strongest_site=float(powers[:n].max()),best_group=float(powers[n:-1].max()),
                fixed_sixteen=float(powers[n+counts.index(16)]),weak=float(powers[-1]))


In [ ]:
POLED_LENGTH_CM = 1.23


LOW_POWER_EFFICIENCY_PER_WATT = 25.9


MEASURED_MAX_WATT = 0.387


MEASURED_MAX_DEPLETION = 0.857


PUMP_LOSS_DB_PER_CM = 0.10


HARMONIC_LOSS_DB_PER_CM = 0.40


FUNDAMENTAL_COUPLING = .45


MISMATCH = .65


WALKOFF_PS_PER_MM = 1.47 / 4.7


PULSE_FWHM_PS = 50.


def rates():
    low, high = np.log(10)/10 * POLED_LENGTH_CM * np.array(
        [PUMP_LOSS_DB_PER_CM, HARMONIC_LOSS_DB_PER_CM])
    difference = high/2-low
    response = np.exp(-high/2)*np.expm1(difference)/difference
    drive_per_sqrt_watt = np.sqrt(LOW_POWER_EFFICIENCY_PER_WATT) / response
    return (low, high), drive_per_sqrt_watt


def single_guide_depletion(g, low, high):
    def rhs(_, state):
        u, h = state
        return [-low*u/2-g*u*h, -high*h/2+g*u*u]
    residual_u, output_h = solve_ivp(rhs, (0., 1.), (1., 0.), method="DOP853",
                                     rtol=1e-11, atol=1e-13).y[:, -1]
    return float(1-residual_u**2/np.exp(-low)), float(output_h**2)


def device_summary():
    loss, factor = rates()
    max_g = factor*np.sqrt(MEASURED_MAX_WATT)
    nominal_depletion, sh = single_guide_depletion(max_g, *loss)
    g_equivalent = brentq(lambda g: single_guide_depletion(g, *loss)[0]-MEASURED_MAX_DEPLETION, .1, 8.)
    empirical_factor = g_equivalent / np.sqrt(MEASURED_MAX_WATT)
    return dict(loss=loss, small_signal_drive_per_sqrt_watt=factor,
                drive_at_largest_demonstrated_power=max_g, nominal_single_guide_depletion=nominal_depletion,
                nominal_single_guide_harmonic_power_fraction=sh, measured_single_guide_depletion=MEASURED_MAX_DEPLETION,
                depletion_equivalent_drive=g_equivalent, effective_high_power_factor=empirical_factor,
                watt_equivalent={str(g): [float((g/factor)**2), float((g/empirical_factor)**2)] for g in (2.,4.,6.,8.,16.)},
                coupling_inv_cm=FUNDAMENTAL_COUPLING/POLED_LENGTH_CM, mismatch_inv_cm=MISMATCH/POLED_LENGTH_CM,
                walkoff_over_FWHM=WALKOFF_PS_PER_MM*10*POLED_LENGTH_CM/PULSE_FWHM_PS)


def planar_model(seed):

    n = 36
    rng = np.random.default_rng(seed)
    edge = rng.normal(0., .12, n - 1)
    depth = rng.normal(size=n)
    def nearest(v, diag):
        return np.diag(diag) + np.diag(v, 1) + np.diag(v, -1)
    base1 = nearest(np.exp(edge), .36*depth)
    base2 = nearest(.46*np.exp(1.6*edge), .65*depth+.16*rng.normal(size=n))
    a = matrix_evolution(base1, 2.1, .045)
    b = matrix_evolution(base2, 2.55, .085).conj().T @ np.eye(n)[:, n//2]
    return dict(a=a, b=b, h1=FUNDAMENTAL_COUPLING*base1,
                h2=FUNDAMENTAL_COUPLING*nearest(.10*np.exp(1.6*edge), np.diag(base2))+MISMATCH*np.eye(n),
                chi=rng.uniform(.5, 1., n), loss=rates()[0])


def pulse(m, x, g, walkoff, time_count=96, steps=100):
    opt = g * m["chi"][:, None, None]
    n_mask = x.shape[1]
    t = np.linspace(-3., 3., time_count, endpoint=False)
    fundamental = np.exp(-2*np.log(2)*t*t)
    u = (m["a"] @ x)[:, :, None] * fundamental[None, None, :]
    v = np.zeros_like(u)
    l1 = -1j*m["h1"]-m["loss"][0]/2*np.eye(len(opt))
    l2 = -1j*m["h2"]-m["loss"][1]/2*np.eye(len(opt))
    dt = t[1]-t[0]
    h = 1./steps
    advance = np.exp(-2j*np.pi*np.fft.fftfreq(time_count, d=dt)*walkoff*h/2)
    input_energy = np.sum(fundamental**2)
    def sh_shift(field):
        return np.fft.ifft(np.fft.fft(field, axis=-1)*advance, axis=-1)
    def rhs(a, b):
        return ((l1 @ a.reshape(len(a), -1)).reshape(a.shape)-1j*opt*a.conj()*b,
                (l2 @ b.reshape(len(b), -1)).reshape(b.shape)-1j*opt*a*a)
    for _ in range(steps):
        if walkoff:
            v = sh_shift(v)
        a, b = rhs(u, v)
        c, d = rhs(u+h*a/2, v+h*b/2)
        e, f = rhs(u+h*c/2, v+h*d/2)
        q, r = rhs(u+h*e, v+h*f)
        u += h*(a+2*c+2*e+q)/6
        v += h*(b+2*d+2*f+r)/6
        if walkoff:
            v = sh_shift(v)
    y = (m["b"].conj() @ v.reshape(len(v), -1)).reshape(n_mask, time_count)
    total = np.sum(abs(u)**2+abs(v)**2, axis=(0, 2))/input_energy
    output = np.sum(abs(y)**2, axis=1)/input_energy
    assert np.all(output <= total+2e-8) and np.all(total <= 1+2e-6)
    return output


def pulse_zero_walkoff(m, x, g, panels=32):
    z, w = np.polynomial.legendre.leggauss(panels)
    t = 3*z
    weights = w*np.exp(-4*np.log(2)*t*t)
    direction = np.exp(-2*np.log(2)*t*t)
    powers = np.array([propagator(m, x, g*q, 100)[0] for q in direction])
    return weights @ powers / weights.sum()


def planar_trial(seed):

    m = planar_model(seed)
    a = matrix_evolution(m['h1'], .5, m['loss'][0]) @ m['a']
    b = matrix_evolution(m['h2'], .5, m['loss'][1]).conj().T @ m['b']
    t = born(m, 32)
    one, group, counts, weak, _, _ = model_masks(a, b.conj()*m['chi'], t, seed)
    all_masks = np.column_stack((one, *group, weak))
    n = len(one)
    rows = []
    mapping = device_summary()
    for g in (mapping["depletion_equivalent_drive"], mapping["drive_at_largest_demonstrated_power"], 6., 8.):
        cw = propagator(m, all_masks, g, 180)[0]
        quasi = pulse_zero_walkoff(m, all_masks, g)
        indexes = [quasi[:n].argmax(), n+1+quasi[n+1:-1].argmax(), n+counts.index(8), len(quasi)-1]
        keep = all_masks[:, indexes]
        walkoff = device_summary()["walkoff_over_FWHM"]
        shift = pulse(m, keep, g, walkoff, time_count=64, steps=80)
        row = dict(drive=g, cw=cw.tolist(), quasi=quasi.tolist(), indexes=[int(i) for i in indexes],
                   nominal_walkoff=shift.tolist())
        if seed in (65379, 66400, 67421, 68442) and g == 8:
            fine = pulse(m, keep, g, walkoff, time_count=128, steps=160)
            verify = pulse(m, keep, g, 0., time_count=128, steps=160)
            strong_walkoff = pulse(m, keep, g, walkoff*3, time_count=128, steps=160)
            error = np.max(abs(shift-fine))
            zero_error = np.max(abs(quasi[indexes]-verify))
            assert error < 4e-4 and zero_error < 4e-5
            row.update(fine_walkoff=fine.tolist(), strong_walkoff=strong_walkoff.tolist(),
                       integration_error=float(error), zero_walkoff_error=float(zero_error))
        rows.append(row)
    return dict(seed=seed, counts=counts, results=rows)


In [ ]:
def local_outputs(g, chi, mismatch, loss, p, steps):
    p = np.asarray(p)
    u = np.array([np.sqrt(p), np.sqrt(1-p)], dtype=complex)
    v = np.zeros_like(u)
    h = 1./steps
    strength = g*np.asarray(chi)[:, None]
    delta = np.asarray(mismatch)[:, None]
    def rhs(a, b):
        return (-1j*strength*a.conj()*b-loss[0]*a/2,
                -1j*strength*a*a-(loss[1]/2+1j*delta)*b)
    for _ in range(steps):
        a, b = rhs(u, v)
        c, d = rhs(u+h*a/2, v+h*b/2)
        e, f = rhs(u+h*c/2, v+h*d/2)
        q, r = rhs(u+h*e, v+h*f)
        u += h*(a+2*c+2*e+q)/6
        v += h*(b+2*d+2*f+r)/6
    assert np.all(np.sum(abs(u)**2+abs(v)**2, axis=0) <= 1+1e-8)
    return v


In [ ]:
GUIDES_PER_COHORT = 128


NOISE_REPEATS = 192


TRAIN_PHASES = 2*np.pi*np.arange(12)/12


NOMINAL_PROBE_COUNTS = (50, 200, 800)


DARK_COUNTS = 5.


PHASE_JITTER = .035


def guide_parameters(rng, near):
    if near:
        return (rng.uniform(.95, 1.05, 2), rng.uniform(-.4, .4, 2),
                rng.uniform(-.2, .2), rng.uniform(.8, 1.), rng.uniform(-.04, .04))
    return (rng.uniform(.85, 1.15, 2), rng.uniform(-2., 2., 2),
            rng.uniform(-.8, .8), rng.uniform(.5, 1.), rng.uniform(-.09, .09))


def calibrate_and_replay(model, scale, offset, counts, rng):
    training_phase = TRAIN_PHASES[None,:]+rng.normal(0, PHASE_JITTER, (NOISE_REPEATS,len(TRAIN_PHASES)))
    training_power = model(training_phase)
    train = rng.poisson(DARK_COUNTS+counts*training_power/scale)
    choice = (train+rng.uniform(0, .5, train.shape)).argmax(axis=1)
    command = TRAIN_PHASES[choice]
    x, w = np.polynomial.hermite.hermgauss(12)
    replay_theta = command[:,None]+np.sqrt(2)*PHASE_JITTER*x
    expected = (model(replay_theta) @ w)/np.sqrt(np.pi)
    replay_phase = command[:,None]+rng.normal(0, PHASE_JITTER,(NOISE_REPEATS,8))
    test = rng.poisson(DARK_COUNTS+counts*model(replay_phase)/scale)
    expected_pump = .5+offset[0]*np.cos(command+offset[1])
    return (expected, test, expected_pump, train, choice)


In [ ]:
def coupler_fields(theta, error):
    t, r = np.sqrt(.5+error), np.sqrt(.5-error)
    step = np.exp(1j*theta)
    return (t+r*step)/np.sqrt(2), (r-t*step)/np.sqrt(2)


def two_guide_trial(seed, near, drive, loss):
    rng = np.random.default_rng(seed)
    chi, mismatch, bias, ratio, error = guide_parameters(rng, near)
    phase_bias = rng.uniform(-np.pi,np.pi)
    rng = np.random.default_rng(seed+(0 if drive==.15 else 52379))
    amp = np.sqrt(.25-error**2)
    p = np.linspace(0,1,1025)
    harmonics = local_outputs(drive, chi, mismatch, loss, p, 180)
    u1, u2 = (CubicSpline(p,h) for h in harmonics)
    scale_weight = np.sqrt(1+ratio**2)
    relamp = ratio*np.exp(1j*bias)
    def model(theta):
        a,b = coupler_fields(theta+phase_bias,error)
        pr = np.clip(abs(a)**2,0,1)
        f0 = np.divide(a,abs(a),out=np.ones_like(a),where=abs(a)>1e-15)
        f1 = np.divide(b,abs(b),out=np.ones_like(b),where=abs(b)>1e-15)
        return abs((u1(pr)*f0**2-relamp*u2(pr)*f1**2)/scale_weight)**2
    sweep = 2*np.pi*np.arange(8192)/8192
    all_power=model(sweep)
    assert np.max(all_power)<=1+1e-10
    best_index=int(all_power.argmax())
    p_best=abs(coupler_fields(sweep[best_index]+phase_bias,error)[0])**2
    physical_single=float(max(abs(harmonics[0,-1])**2,ratio**2*abs(harmonics[1,0])**2)/(1+ratio**2))
    endpoint=float(model(np.array([0.,np.pi])-phase_bias).max())
    equal=float(model(np.array([np.pi/2.,3*np.pi/2.])-phase_bias).mean())
    count_scale=.5*np.tanh(drive)**2
    settings=[]
    example=None
    verification=None
    for count in NOMINAL_PROBE_COUNTS:
        expected, readback, choices_pump, train, choice=calibrate_and_replay(
            model,count_scale,(amp,phase_bias),count,rng)
        settings.append(dict(counts=count,selected_mean_power=float(expected.mean()),
                             detector_replay_mean_power=float((readback.mean()-DARK_COUNTS)*count_scale/count),
                             selected_within_95pct=float(np.mean(expected>=.95*all_power.max())),
                             selected_near_one=float(np.mean(np.maximum(choices_pump,1-choices_pump)>=.85)),
                             selected_near_split=float(np.mean((choices_pump>=.30)&(choices_pump<=.70))),
                             beats_best_single=float(np.mean(expected>physical_single*(1+1e-8)))))
        if near and seed==473029 and count==200:
            theta=np.linspace(0,2*np.pi,361)
            example=dict(seed=seed,drive=drive,phases=theta.tolist(),collected_power=model(theta).tolist(),
                         train_phases=TRAIN_PHASES.tolist(),first_train=train[0].tolist(),
                         first_selected_index=int(choice[0]),first_replay_counts=readback[0].tolist(),
                         first_selected_true_power=float(expected[0]),scale=float(count_scale),
                         physical_single=physical_single)
            samples=np.array([0.,.57,1.4,2.3,3.19,4.6,5.7])
            a,b=coupler_fields(samples+phase_bias,error)
            initial=np.array([a,b,np.zeros_like(a),np.zeros_like(a)])
            strength=drive*np.asarray(chi)[:,None]
            delta=np.asarray(mismatch)[:,None]
            def rhs(_, flat):
                pump,sh=flat.reshape(2,2,-1)
                return np.array([-1j*strength*pump.conj()*sh-loss[0]*pump/2,
                                 -1j*strength*pump**2-(loss[1]/2+1j*delta)*sh]).ravel()
            field=solve_ivp(rhs,(0,1),initial.ravel(),method='DOP853',rtol=2e-11,
                            atol=4e-13).y[:,-1].reshape(4,-1)[2:]
            absolute_power=abs((field[0]-relamp*field[1])/scale_weight)**2
            verification=float(np.max(abs(absolute_power-model(samples))))
            assert verification<1e-8
    row=dict(seed=seed,cohort='near_matched' if near else 'wider_errors',drive=drive,
             true_optimal_p=float(p_best),oracle_power=float(all_power[best_index]),best_single=physical_single,
             accessible_single_like_endpoint=endpoint,equal_split=equal,settings=settings)
    return row,example,verification


In [ ]:
def collect_dense_reference():
    rng = np.random.default_rng(5611240)
    lower = lower_bound()
    upper = upper_bound()
    cases = ((2, 10, 8), (8, 24, 8), (16, 24, 8), (36, 24, 10), (64, 12, 8),
             (128, 32, 10), (512, 24, 8), (1024, 12, 8))
    rows = []
    for n, repeats, starts in cases:
        print(f"  N = {n}", flush=True)
        raw = [dense_reference_trial(n, rng, lower, n <= 64, starts) for _ in range(repeats)]
        summary = {}
        for key in raw[0]:
            x = np.array([r[key] for r in raw])
            summary[key] = dict(mean=float(x.mean()), standard_error=float(x.std(ddof=1) / np.sqrt(repeats)))
        rows.append(dict(modes=n, repetitions=repeats, random_starts=starts, raw=raw, summary=summary))
    return dict(analytic_lower=lower, analytic_upper=upper, rows=rows)


def collect_sparse():
    with redirect_stdout(StringIO()):
        raw = sparse_scatterers(np.random.default_rng(12259321))
    return [dict(n=r["modes"], k=r["scatterers"],
                 exact_ensemble_mean=finite_size_mean(r["modes"], r["scatterers"]),
                 simulated_mean=r["mean_enhancement_per_mode"],
                 leading_asymptote=r["leading_log_term"]) for r in raw]


def crossing(x, y):
    indices = np.flatnonzero(y <= 0.5)
    if not len(indices) or indices[0] == 0:
        return None
    i = indices[0]
    fraction = (y[i-1] - 0.5) / (y[i-1] - y[i])
    return float(x[i-1] * (x[i] / x[i-1]) ** fraction)


def collect_crossover():
    output = []
    rng_boot = np.random.default_rng(12011)
    for n in (48, 96, 192, 384):
        print(f"  N = {n}", flush=True)
        rng = np.random.default_rng(14751 + n)
        ratios = (.0625, .125, .25, .5, 1., 2., 4., 8., 16., 32., 64.)
        if n == 384:
            ratios = (.125, .5, 2., 8., 32.)
        pairs = 10 if n == 384 else 18
        raw = []
        for ratio in ratios:
            k = max(2, round(n * ratio))
            cert = n == 48 and ratio in (.5, 2., 8.)
            raw.append([crossover_trial(n, k, rng, cert) for _ in range(pairs)])
        ratios = np.array([max(2, round(n * ratio)) / n for ratio in ratios])
        draws = np.array([[[entry["strongest_one_fraction"] for entry in run]
                           for run in condition] for condition in raw])
        ids = rng_boot.integers(pairs, size=(6000, len(raw), pairs))
        boot = draws[np.arange(len(raw))[None, :, None], ids].mean(axis=2)
        for j, law in enumerate(("gaussian", "clipped_1p5", "unit_magnitude")):
            values = draws[:, :, j].mean(axis=1)
            cuts = [cut for sample in boot[:, :, j] if (cut := crossing(ratios, sample)) is not None]
            output.append(dict(n=n, distribution=law, ratios=ratios.tolist(), mean=values.tolist(),
                               ci_low=np.quantile(boot[:, :, j], .025, axis=0).tolist(),
                               ci_high=np.quantile(boot[:, :, j], .975, axis=0).tolist(),
                               half=crossing(ratios, values), bootstrap_half_available=len(cuts)/len(boot),
                               bootstrap_half_ci=np.quantile(cuts, [.025, .975]).tolist() if cuts else []))
    return output


def collect_passive_collection():
    n, repeats = 192, 9
    rng = np.random.default_rng(5413 + 3 * n)
    for ratio in (.125, .25, .5, 1., 2.):
        k = int(n * ratio)
        for extra in (1., 1.5):
            raw = {"speckled": [], "flat": []}
            for _ in range(repeats):
                a, d = make_propagation(n, k, rng, extra)
                chi = rng.uniform(.5, 1., k)
                b = draw_target(k, rng)
                x = np.exp(2j * np.pi * rng.random(n)) / np.sqrt(n)
                for law, port in zip(raw, b):
                    raw[law].append(weak_test(a, d, port, chi, x, rng))
    return [dict(n=n, k=k, d=d, extras=extra, distribution=law, samples=repeats,
                 summary=summarize_passive_trials(rows), raw=rows) for law, rows in raw.items()]


def collect_passive_small():
    n = 64
    g = 8.
    rng = np.random.default_rng(612834)
    start_rng = np.random.default_rng(746650)
    rows = []
    for _ in range(14):
        a, d = make_propagation(n, n, rng, 1.)
        chi = rng.uniform(.5, 1., n)
        b, _ = draw_target(n, rng)
        strongest = np.argsort(-abs(b * chi))
        masks = [mask_for_outputs(a, b, strongest[:c]) for c in (1, 2, 4, 8, 16)]
        matrix = a.T @ ((b.conj() * chi)[:, None] * a) * d / np.sqrt(n * CHI2)
        leading, _ = leading_mask(matrix, rng)
        _, weak = optimal_values(matrix, [masks[0], leading])
        random_masks = np.exp(2j * np.pi * start_rng.random((8, n))) / np.sqrt(n)
        _, expanded_weak = optimal_values(matrix, [weak, leading, *masks, *random_masks])
        weak_power = max(conversion(a, b, chi, g, mask)[0] for mask in (weak, expanded_weak))
        powers = np.array([conversion(a, b, chi, g, x)[0] for x in masks])
        starts = [masks[0], masks[powers.argmax()], weak, expanded_weak]
        opt, total, intercepted = saturating_best(a, b, chi, g, starts)
        rows.append(dict(one=float(powers[0]), simple_multi=float(powers.max()),
                         weak_mask_at_drive=float(weak_power),
                         optimized=opt, material_conversion=total, intercepted=intercepted))
    return rows


def collect_two_guides():
    loss, _ = rates()
    hi = device_summary()["depletion_equivalent_drive"]
    rows, curves, errors = [], [], []
    for near in (True, False):
        for drive in (.15, hi):
            print(f"  {'Near-matched' if near else 'Wider errors'}, g = {drive:.3f}", flush=True)
            for i in range(GUIDES_PER_COHORT):
                seed = 473029 + 1031*i + (0 if near else 164093)
                row, curve, error = two_guide_trial(seed, near, drive, loss)
                rows.append(row)
                if curve is not None:
                    curves.append(curve)
                    errors.append(error)
    return dict(rows=rows, curves=curves, solver_errors=errors)


def collect_sequence(task, cases, report_every=8):
    rows = []
    for i, case in enumerate(cases, 1):
        rows.append(task(case))
        if i % report_every == 0 or i == len(cases):
            print(f"  {i}/{len(cases)}", flush=True)
    return rows


def calculate_simulations():
    batch = {
        "dense_reference": collect_dense_reference,
        "dense_fresh": lambda: collect_sequence(lambda t: dense_fresh_trial(*t), theory_cases("dense")),
        "dense_exponent": bound_report,
        "sparse": collect_sparse,
        "crossover": collect_crossover,
        "passive_collection": collect_passive_collection,
        "passive_groups": lambda: collect_sequence(lambda t: passive_group_trial(*t), theory_cases("passive")),
        "passive_screened": lambda: collect_sequence(screened_passive_trial,
                                                     [893817+1021*i for i in range(24)]),
        "passive_small": collect_passive_small,
        "two_guides": collect_two_guides,
        "square_array": lambda: collect_sequence(optical_task, [row for row in optical_cases("waveguide")
                                                                       if row[1] == "new" and row[3] == .45]),
        "coupling": lambda: collect_sequence(coupling_trial,
                                            [(65379+1021*i, h, d) for h,d in ((.45,.45),(1.6,.45),(.45,1.6),(1.6,1.6))
                                             for i in range(16)]),
        "diffraction": lambda: collect_sequence(optical_task, [row for row in optical_cases("diffraction")
                                                                       if row[1] == "new"]),
        "planar": lambda: collect_sequence(planar_trial, [65379+1021*i for i in range(16)], report_every=1),
    }
    output = {}
    for name, compute in batch.items():
        start = perf_counter()
        print(name, flush=True)
        output[name] = compute()
        print(f"  {perf_counter()-start:.1f} seconds", flush=True)
    return output


In [ ]:
BLUE, ORANGE, GREEN, PURPLE, GRAY = "#24688e", "#ba562c", "#477d57", "#745b92", "#59636c"


LAWS = {"gaussian": ("Gaussian", BLUE), "clipped_1p5": ("Clipped Gaussian", ORANGE),
        "unit_magnitude": ("Equal magnitudes", GREEN)}


def style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 8.5, "axes.labelsize": 8.5,
        "axes.titlesize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": .7,
        "legend.frameon": False, "legend.fontsize": 7.3,
        "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.facecolor": "white",
        "axes.labelpad": 4, "xtick.direction": "out", "ytick.direction": "out",
    })


def panels(rows=2, height=5.45):
    fig, ax = plt.subplots(rows, 2, figsize=(7.25, height), layout="constrained")
    flat = ax.ravel()
    for label, panel in zip("abcdef", flat):
        panel.set_title(label, loc="left", fontweight="bold")
    return fig, flat


def diagram(ax):
    ax.set(xlim=(0,1), ylim=(0,1))
    ax.axis("off")


def box(ax, x, y, w, h, label, color=BLUE):
    rect = FancyBboxPatch((x,y),w,h, boxstyle="round,pad=0.015",
                           edgecolor=color, facecolor="white", linewidth=.85)
    ax.add_patch(rect)
    ax.text(x+w/2,y+h/2,label,color=color,ha="center",va="center")


def arrow(ax, point_a, point_b, color=GRAY, alpha=1):
    ax.annotate("", xy=point_b, xytext=point_a,
                arrowprops=dict(arrowstyle="->",color=color,lw=.8,alpha=alpha,
                                shrinkA=1,shrinkB=1))


def sem(x):
    x = np.asarray(x)
    return x.std(axis=0, ddof=1)/np.sqrt(len(x))


def ratio_ci(x, y):
    x, y = np.array(x), np.array(y)
    draws = np.random.default_rng(10733).integers(len(x),size=(10000,len(x)))
    ratio = x[draws].mean(axis=1)/y[draws].mean(axis=1)
    return np.array([x.mean()/y.mean(), *np.quantile(ratio,[.025,.975])])


def interval_points(ax, x, records, color=BLUE, marker="o"):
    mean, lo, hi = np.asarray(records).T
    ax.errorbar(x,mean,yerr=[np.maximum(0,mean-lo),np.maximum(0,hi-mean)],
                linestyle="none",marker=marker,color=color,capsize=2.5,markersize=4)


def paired_plot(ax, values, labels, ylabel, colors):
    values = np.array(values)
    x = np.arange(values.shape[1])
    for row in values:
        ax.plot(x,row,color=GRAY,alpha=.19,lw=.65)
    for i, color in enumerate(colors):
        ax.scatter(np.full(len(values),i),values[:,i],s=9,color=color,alpha=.38,zorder=3)
        ax.errorbar(i,values[:,i].mean(),yerr=sem(values[:,i]),fmt="D",color=color,
                    markeredgecolor="white",ms=5,capsize=2,zorder=4)
    ax.set(xticks=x,xticklabels=labels,ylabel=ylabel)
    ax.margins(x=.1)


def by_drive(record, drive):
    return next(row for row in record["results"] if np.isclose(row["drive"],drive))


def figure1(data):
    fig, axs = panels(rows=3, height=7.85)
    ax = axs[0]
    diagram(ax)
    ax.text(.5,.97,"Phase-only control of second-harmonic light",ha="center",va="top")
    box(ax,.04,.36,.18,.24,"Input\nphases")
    for y in (.18,.32,.48,.64,.78):
        ax.add_patch(Circle((.49,y),.024,facecolor=GREEN,edgecolor="white",lw=.75,zorder=3))
        arrow(ax,(.22,.48),(.466,y),BLUE,.73)
        arrow(ax,(.514,y),(.78,.48),ORANGE,.73)
    box(ax,.78,.36,.18,.24,"One\nreceiver",ORANGE)
    for x, label, color in ((.23,"Pump",BLUE),(.47,"Nonlinear sites",GREEN),(.80,"Second harmonic",ORANGE)):
        ax.text(x,.06,label,ha="center",color=color,fontsize=7.3)

    ax = axs[1]
    diagram(ax)
    ax.text(.5,.91,"Shared-scatterer model",ha="center",weight="medium")
    ax.text(.5,.74,r"$u_k=(Ax)_k,\qquad |x_j|=N^{-1/2}$",ha="center",fontsize=10)
    ax.text(.5,.57,r"$q=(KN)^{-1/2}\sum_k z_k u_k^2=x^{\mathsf{T}}Tx$",ha="center",fontsize=10)
    ax.text(.5,.44,r"$T=A^{\mathsf{T}}\,\mathrm{diag}(z)\,A/\sqrt{KN}$",ha="center")
    box(ax,.05,.14,.37,.17,"Single bright site",GREEN)
    box(ax,.58,.14,.37,.17,"Collective response",ORANGE)

    ax = axs[2]
    n = 1024
    rows = [r for r in data["dense_fresh"] if r["n"]==n]
    values = [[r[k]/(n*r["background"]) for k in ("projected","corrected","best")] for r in rows]
    paired_plot(ax,values,["Projected\nTakagi","Analytic\nupdate","Numerical\nmask"],
                r"Enhancement per input, $\eta/N$",[BLUE,GREEN,ORANGE])
    ax.axhline(np.pi**2/8,color=GRAY,ls="--",lw=.8,label=r"Projected Takagi, $\pi^2/8$")
    ax.set_title(r"Independent Gaussian entries, $N=1{,}024$")
    ax.legend(loc="upper left")

    ax = axs[3]
    r = data["dense_reference"]
    sizes = [a["modes"] for a in r["rows"]]
    lo = r["analytic_lower"]["asymptotic_enhancement_per_mode"]
    hi = r["analytic_upper"]["continuous_enhancement_per_mode"]
    ax.axhspan(lo,hi,color=GREEN,alpha=.12,label="Asymptotic bounds on optimum")
    ax.errorbar(sizes,[a["summary"]["feasible"]["mean"] for a in r["rows"]],
                yerr=[a["summary"]["feasible"]["standard_error"] for a in r["rows"]],
                fmt="o-",ms=4,lw=1,capsize=2,color=BLUE,label="Feasible finite-size mean")
    ax.axhline(np.pi**2/8,color=GRAY,ls="--",lw=.8)
    ax.set(xscale="log",xlabel="Input modes, $N$",ylabel=r"Enhancement per input, $\eta/N$")
    ax.set_title("Independent Gaussian entries")
    ax.legend(loc="lower right")
    optical_gain_panels(axs[4], axs[5], data["optical"])
    return fig


def optical_gain_panels(cumulative, paired, records):
    curves = (
        ("one_step_update", "Analytic update", BLUE, "-"),
        ("field_gradient_matched_rms", "Matched field step", GREEN, "--"),
        ("intensity_gradient_fixed_rms", "Fixed intensity step", PURPLE, ":"),
    )
    gains = {}
    for key, label, color, linestyle in curves:
        values = receiver_gain(records, key)
        gains[key] = values
        sorted_gains = np.sort(values)
        cumulative.step(
            np.r_[sorted_gains[0], sorted_gains, 40],
            np.r_[0, np.arange(1, len(values) + 1) / len(values), 1],
            where="post", lw=1.1, color=color, linestyle=linestyle, label=label,
        )
    cumulative.axvline(0, color=GRAY, lw=.65)
    cumulative.set(xlim=(-1.5, 40), ylim=(0, 1.02),
                   xticks=[0, 10, 20, 30, 40],
                   xlabel="Predicted gain over projected Takagi (%)",
                   ylabel="Fraction of receivers",
                   title="Optical receiver tensors")
    cumulative.legend(loc="lower right")

    field = gains["field_gradient_matched_rms"]
    analytic = gains["one_step_update"]
    paired.scatter(field, analytic, s=7, color=BLUE, alpha=.42, edgecolors="none")
    paired.plot([-1.5, 40], [-1.5, 40], "--", color=GRAY, lw=.75)
    paired.set(xlim=(-1.5, 40), ylim=(-1.5, 40), xticks=[0, 10, 20, 30, 40],
               yticks=[0, 10, 20, 30, 40], xlabel="Gradient step gain (%)",
               ylabel="Analytic update gain (%)", title="Paired comparison at equal step size")
    wins = count_receiver_wins(records, "field_gradient_matched_rms")
    paired.text(.96, .065, f"Analytic update higher at {wins} of {len(records)}",
                transform=paired.transAxes, ha="right", va="bottom", fontsize=7)


def figure2(data):
    fig, axs = panels()
    ax = axs[0]
    diagram(ax)
    ax.text(.5,.93,"Gaussian collection weights",ha="center")
    ax.add_patch(Rectangle((.35,.26),.30,.49,color=GRAY,alpha=.10))
    arrow(ax,(.05,.48),(.95,.48))
    ax.text(.18,.64,"Brightest-site\nregime",ha="center",color=BLUE)
    ax.text(.50,.64,"Crossover",ha="center",color=GRAY)
    ax.text(.81,.64,"Collective\nresponse",ha="center",color=ORANGE)
    for x,label in ((.18,r"$\ll 1$"),(.50,r"$\asymp 1$"),(.81,r"$\gg 1$")):
        ax.text(x,.35,label,ha="center")
    ax.text(.50,.13,r"$K/(N\log K)$",ha="center",fontsize=10)

    ax=axs[1]
    rows = [r for r in data["sparse"] if 2*r["k"]==r["n"]]
    n = [r["n"] for r in rows]
    ax.plot(n,[r["leading_asymptote"] for r in rows],ls="--",color=GRAY,label="Leading asymptote")
    ax.plot(n,[r["exact_ensemble_mean"] for r in rows],color=GREEN,label="Exact finite mean")
    ax.scatter(n,[r["simulated_mean"] for r in rows],s=24,color=BLUE,facecolor="white",zorder=4,
               label="Simulated mean")
    ax.set(xscale="log",xlabel="Input modes, $N$",ylabel=r"Enhancement per input, $\eta/N$",title="Brightest-weight site at $K=N/2$")
    ax.legend(loc="lower right")

    ax=axs[2]
    for r in [x for x in data["crossover"] if x["n"]==192]:
        label,color=LAWS[r["distribution"]]
        ax.plot(r["ratios"],r["mean"],".-",color=color,label=label)
        ax.fill_between(r["ratios"],r["ci_low"],r["ci_high"],color=color,alpha=.13)
        ax.errorbar(r["half"],.5,xerr=[[r["half"]-r["bootstrap_half_ci"][0]],
                                     [r["bootstrap_half_ci"][1]-r["half"]]],fmt="o",
                    color=color,markerfacecolor="white",ms=4,capsize=2)
    ax.axhline(.5,color=GRAY,ls="--",lw=.7)
    ax.set(xscale="log",ylim=(0,1.05),xlabel=r"Scatterers per input, $K/N$",
           ylabel="One-site / best found intensity",title="$N=192$")
    ax.legend()

    ax=axs[3]
    rows={r["distribution"]:r for r in data["passive_collection"]}
    pair=np.column_stack([[a["single_over_best"] for a in rows[name]["raw"]]
                          for name in ("speckled","flat")])
    paired_plot(ax,pair,["Uneven","Equal"],"One-site / best found intensity",[BLUE,GREEN])
    ax.set(title="$N=192$, $K=384$, $D=576$",ylim=(0,1.01),xlabel="Collection weights")
    return fig


def figure3(data):
    fig,axs=panels()
    ax=axs[0]
    p=np.linspace(0,1,180)
    for g,color in ((.15,BLUE),(2.,GREEN),(8.,ORANGE)):
        ax.plot(p,np.sqrt(p)*np.tanh(g*np.sqrt(p))/g,color=color,label=f"$g={g:g}$")
    ax.set(xlabel="Local pump fraction",ylabel=r"Harmonic amplitude magnitude / $g$",
           title="Phase-matched conversion")
    ax.legend()

    ax=axs[1]
    rows=data["passive_screened"]
    sizes=rows[0]["tested_group_sizes"]
    powers=100*np.array([r["groups"] for r in rows])
    ref=100*np.mean([r["screened"] for r in rows])
    ax.axhline(ref,ls="--",lw=.85,color=GRAY,label="Best screened one-site mean")
    ax.errorbar(sizes,powers.mean(axis=0),yerr=sem(powers),fmt="o-",ms=3,capsize=2,color=BLUE,
                label="Mean by group size")
    ax.set(xscale="log",xticks=[1,4,16,64,256],xticklabels=["1","4","16","64","256"],
           xlabel="Group size",ylabel="Collected power (% of launch)",title=r"$N=K=D=8{,}192$, $g=8$")
    ax.legend(loc="upper left")

    ax=axs[2]
    rows=[r for r in data["passive_groups"] if r["n"]==8192 and r["d"]==8192]
    sizes=rows[0]["counts"]
    img=np.array([np.bincount(np.asarray([r["power"][key] for r in rows]).argmax(axis=1),
                              minlength=len(sizes)) for key in ("0.15","8.0","12.0")])
    heat=ax.imshow(img,aspect="auto",cmap="Blues",vmin=0,vmax=len(rows))
    for i,j in np.argwhere(img>0):
        ax.text(j,i,str(img[i,j]),ha="center",va="center",color="white" if img[i,j]>12 else GRAY)
    ax.set(xticks=range(len(sizes)),xticklabels=sizes,yticks=[0,1,2],yticklabels=["0.15","8","12"],
           xlabel="Best tested group size",ylabel="Drive, $g$",title="Group choices in 24 geometries")
    fig.colorbar(heat,ax=ax,shrink=.67,label="Geometries")

    ax=axs[3]
    rows=data["passive_small"]
    values=100*np.array([[r[k] for k in ("one","weak_mask_at_drive","simple_multi","optimized")] for r in rows])
    paired_plot(ax,values,["One\nsite","Weak-drive\nmask","Group\nmask","Direct\noptimizer"],
                "Collected power (% of launch)",[GRAY,BLUE,GREEN,ORANGE])
    ax.set_title("$N=64$, $g=8$")
    return fig


def guide_optimum(g, ratio):
    def slope(p):
        x=g*np.sqrt(p)
        return g if x==0 else (g/2)*(np.tanh(x)/x+1/np.cosh(x)**2)
    if slope(1)>=ratio*g:
        return 1.
    return brentq(lambda p: slope(p)-ratio*slope(1-p),0,1)


def figure4(data):
    fig,axs=panels()
    rows=data["two_guides"]["rows"]
    ax=axs[0]
    diagram(ax)
    box(ax,.27,.39,.16,.19,"Pump\ncoupler")
    box(ax,.55,.62,.15,.16,"Guide 1",GREEN)
    box(ax,.55,.21,.15,.16,"Guide 2",GREEN)
    box(ax,.80,.39,.166,.19,"Harmonic\ncombiner",ORANGE)
    for a,b,color in (((.04,.545),(.27,.545),BLUE),((.04,.425),(.27,.425),BLUE),
                      ((.43,.52),(.55,.70),BLUE),((.43,.45),(.55,.29),BLUE),((.70,.70),(.80,.52),ORANGE),
                      ((.70,.29),(.80,.46),ORANGE)):
        arrow(ax,a,b,color)
    ax.text(.14,.59,r"$1/\sqrt{2}$",ha="center",color=BLUE)
    ax.text(.14,.34,r"$e^{i\theta}/\sqrt{2}$",ha="center",color=BLUE)
    ax.text(.46,.66,r"$p$",ha="center",color=BLUE)
    ax.text(.46,.27,r"$1-p$",ha="center",color=BLUE)
    ax.text(.50,.08,"Two-guide numerical example",ha="center")

    ax=axs[1]
    r=.90
    g=np.linspace(.025,2.,180)
    critical=brentq(lambda x: (np.tanh(x)/x+1/np.cosh(x)**2)/2-r,.05,3)
    ax.plot(g,[guide_optimum(x,r) for x in g],color=BLUE)
    ax.axvline(critical,ls="--",lw=.8,color=GRAY)
    ax.annotate(f"$g={critical:.3f}$",(critical,.58),xytext=(.53,.55),
                arrowprops=dict(arrowstyle="-",color=GRAY,lw=.7),fontsize=8)
    ax.set(xlabel="Drive, $g$",ylabel="Optimal pump fraction in guide 1",
           ylim=(.46,1.04),title="Ideal guides, field ratio 0.90")

    ax=axs[2]
    for curve,color in zip(data["two_guides"]["curves"],[BLUE,ORANGE]):
        phase=np.asarray(curve["phases"])/np.pi
        reference=curve["physical_single"]
        ratio=np.asarray(curve["collected_power"])/reference
        probes=np.asarray(curve["train_phases"])/np.pi
        detector=((np.asarray(curve["first_train"])-DARK_COUNTS)*curve["scale"]/200)/reference
        i=curve["first_selected_index"]
        replay=curve["first_selected_true_power"]/reference
        ax.plot(phase,ratio,color=color,label=f'$g={curve["drive"]:.2f}$')
        other=np.arange(len(probes))!=i
        ax.scatter(probes[other],detector[other],s=12,color=color,facecolor="white",zorder=4)
        ax.plot([probes[i]]*2,[detector[i],replay],color=color,ls=":",lw=.8)
        ax.scatter(probes[i],detector[i],s=22,color=color,facecolor="white",marker="D",zorder=5)
        ax.scatter(probes[i],replay,s=26,color=color,marker="^",edgecolor="white",linewidth=.45,zorder=5)
    ax.axhline(1,ls="--",lw=.75,color=GRAY)
    ax.set(xlabel=r"Command phase / $\pi$",ylabel="Power / best single guide",title="One geometry, 12 noisy probes")
    ax.legend(loc="upper left",handlelength=1.5)

    ax=axs[3]
    for group,curve in enumerate(data["two_guides"]["curves"]):
        subset=[r for r in rows if r["cohort"]=="near_matched" and r["drive"]==curve["drive"]]
        base=[r["best_single"] for r in subset]
        for method,vals,col,marker,delta in (
            ("12 probes",[r["settings"][1]["selected_mean_power"] for r in subset],BLUE,"o",-.1),
            ("Known equal split",[r["equal_split"] for r in subset],GREEN,"^",0),
            ("Dense sweep",[r["oracle_power"] for r in subset],ORANGE,"s",.1)):
            interval_points(ax,[group+delta],[ratio_ci(vals,base)],col,marker)
            if group==0:
                ax.plot([],[],linestyle="none",marker=marker,color=col,label=method,ms=4)
    ax.axhline(1,ls="--",lw=.75,color=GRAY)
    ax.set(xlim=(-.35,1.35),xticks=[0,1],xticklabels=["Weak drive","High drive"],
           ylabel="Power / best single guide",title="128 near-matched geometries")
    ax.legend(loc="upper left")
    return fig


def figure5(data):
    fig,axs=panels()
    square=data["square_array"]
    drives=[r["drive"] for r in square[0]["results"]]
    ax=axs[0]
    def fixed(row):
        return row["groups"][row["counts"].index(16)]
    for name,extract,col in (("Site-matched",lambda r:r["single"],GRAY),
                             ("16-site group",fixed,GREEN),
                             ("Better of two weak-drive masks",lambda r:max(r["weak"],r["old_weak"]),BLUE)):
        z=100*np.array([[extract(by_drive(r,g)) for r in square] for g in drives])
        ax.errorbar(drives,z.mean(axis=1),yerr=sem(z.T),fmt="o-",ms=3,capsize=2,label=name,color=col)
    z=100*np.array([by_drive(r,8)["optimized"] for r in square])
    ax.errorbar(8,z.mean(),yerr=sem(z),fmt="D",color=ORANGE,ms=4,capsize=2,label="Direct optimization")
    ax.set(xlabel="Drive, $g$",ylabel="Collected power (% of launch)",title="Moderate hopping")
    ax.legend(loc="upper left")

    ax=axs[1]
    ablated=[by_drive(r,8) for r in square]
    baseline=[r for r in data["coupling"] if (r["hopping"],r["disorder"])==(.45,.45)]
    hopping=[r for r in data["coupling"] if (r["hopping"],r["disorder"])==(1.6,.45)]
    born=[r["born_group"][np.argmax(r["groups"])] for r in ablated]
    points=[ratio_ci([r["best_group"] for r in rows],[r["weak"] for r in rows])
            for rows in (baseline,hopping)]
    interval_points(ax,[0,1,2],[points[0],ratio_ci(born,[r["born_weak"] for r in ablated]),points[1]])
    ax.axhline(1,ls="--",lw=.75,color=GRAY)
    ax.set(xticks=[0,1,2],xticklabels=["Moderate\nhopping","Undepleted\nreplay","Higher\nhopping only"],
           ylabel="Group / weak-drive reference",title="Matched comparisons at $g=8$")

    ax=axs[2]
    rows=[by_drive(r,16) for r in data["diffraction"]]
    z=100*np.array([[r["single"],r["groups"][r["counts"].index(4)],max(r["groups"]),
                    max(r["weak"],r["old_weak"]),r["optimized"]] for r in rows])
    paired_plot(ax,z,["One\nsite","Four\nsites","Selected\ngroup","Weak-drive\nreference","Direct\nsearch"],
                "Collected power (% of launch)",[GRAY,GREEN,ORANGE,BLUE,PURPLE])
    ax.set_title("Shared-path diffraction at $g=16$")

    ax=axs[3]
    rows=data["planar"]
    chosen=[0,1,len(rows[0]["results"])-1]
    out=[]
    for i in chosen:
        power=np.array([r["results"][i]["nominal_walkoff"] for r in rows])
        out.append(ratio_ci(power[:,1],power[:,3]))
    interval_points(ax,[0,1,2],out)
    ax.axhline(1,ls="--",lw=.75,color=GRAY)
    ax.set(xticks=[0,1,2],xticklabels=["$g=1.65$\nDepletion fit","$g=3.30$\nWeak-drive fit","$g=8$\nStress test"],
           ylabel="Group / weak-drive pulse energy",title="Planar waveguide array")
    return fig


def supplement1(data):
    fig,axs=panels()
    dense=data["dense_reference"]["rows"]
    for key,col,name in (("projected",BLUE,"Takagi mask"),("analytic_update",GREEN,"Analytic update"),
                         ("feasible",ORANGE,"Numerical mask")):
        sizes = [r["modes"] for r in dense]
        means = [r["summary"][key]["mean"] for r in dense]
        axs[0].plot(sizes,means,"o-",ms=3,color=col,label=name)
    axs[0].set(xscale="log",xlabel="$N$",ylabel=r"Mean $\eta/N$")
    axs[0].legend()
    ax=axs[1]
    for r in dense:
        if "finite_instance_upper" in r["raw"][0]:
            ax.scatter([r["modes"]]*len(r["raw"]),
                       [q["finite_instance_upper"]-q["feasible"] for q in r["raw"]],s=9,alpha=.35,color=BLUE)
    ax.set(xscale="log",xlabel="$N$",ylabel=r"Upper bound minus achieved $\eta/N$")
    ax=axs[2]
    rows=data["dense_fresh"]
    x=np.array([r["n"] for r in rows])
    y=np.array([r["background"]*r["n"]/2 for r in rows])
    ax.scatter(x,y,alpha=.5,s=10,color=BLUE,label="Fixed-tensor mean / ensemble mean")
    ax.axhline(1,color=GRAY,ls="--",lw=.75)
    ax.set(xscale="log",xlabel="$N$",ylabel="Background ratio")
    ticks=np.unique(x)
    ax.set_xticks(ticks,labels=[f"{n:,}" for n in ticks])
    ax.set_xticks([],minor=True)
    ax.legend()
    ax=axs[3]
    ax.scatter([r["background"] for r in rows],[r["background_mc"] for r in rows],s=12,color=BLUE,alpha=.55)
    x=np.array([r["background"] for r in rows]); ax.plot([x.min(),x.max()],[x.min(),x.max()],"--",lw=.75,color=GRAY)
    ax.set(xlabel="Analytic phase-random mean",ylabel="Monte Carlo phase-random mean",
           xscale="log",yscale="log")
    return fig


def supplement2(data):
    fig,axs=panels()
    for ax,n in zip(axs,(48,96,192,384)):
        for r in [r for r in data["crossover"] if r["n"]==n]:
            label,c=LAWS[r["distribution"]]
            ax.plot(r["ratios"],r["mean"],".-",color=c,label=label)
            ax.fill_between(r["ratios"],r["ci_low"],r["ci_high"],color=c,alpha=.13)
        ax.axhline(.5,ls="--",color=GRAY,lw=.7)
        ax.set(xscale="log",xlim=(.05,80),ylim=(0,1.06),title=f"$N={n}$",xlabel="$K/N$",
               ylabel="One-site / best found intensity")
    axs[0].legend()
    return fig


def supplement3(data):
    fig,axs=panels()
    for ax,(n,extra) in zip(axs,((2048,1.),(2048,1.5),(8192,1.),(8192,1.5))):
        rows=[r for r in data["passive_groups"] if r["n"]==n and r["d"]==int(n*extra)]
        for key,c in (("0.15",BLUE),("2.0",GREEN),("5.0",PURPLE),("8.0",ORANGE),("12.0",GRAY)):
            z=100*np.array([r["power"][key] for r in rows])
            ax.plot(rows[0]["counts"],z.mean(axis=0),"o-",ms=2.5,color=c,label=f"$g={float(key):g}$")
        ax.set(xscale="log",yscale="log",ylim=(.00018,11),xlabel="Group size",ylabel="Collected power (% of launch)",
               title=f"$N=K={n:,}$  /  $D/N={extra:g}$")
    axs[0].legend(ncol=2)
    return fig


def supplement4(data):
    fig,axs=panels()
    ax=axs[0]
    keys=((.45,.45),(1.6,.45),(.45,1.6),(1.6,1.6))
    pts=[]
    for h,d in keys:
        rows=[r for r in data["coupling"] if (r["hopping"],r["disorder"])==(h,d)]
        pts.append(ratio_ci([r["best_group"] for r in rows],[r["weak"] for r in rows]))
    interval_points(ax,range(4),pts)
    ax.axhline(1,lw=.75,ls="--",color=GRAY)
    ax.set(xticks=range(4),xticklabels=["Baseline","Higher\nhopping","Higher\ndisorder","Both higher"],
           ylabel="Group / weak-drive reference",title="Hopping and disorder varied separately")
    model=data["diffraction"]
    ax=axs[1]
    pts=[]
    for g in (.2,8,16):
        rows=[by_drive(r,g) for r in model]
        multisite=[max(p for n,p in zip(r["counts"],r["groups"]) if n>1) for r in rows]
        pts.append(ratio_ci(multisite,[r["single"] for r in rows]))
    interval_points(ax,[.2,8,16],pts)
    ax.axhline(1,lw=.75,ls="--",color=GRAY)
    ax.set(xlabel="Drive, $g$",xticks=[.2,8,16],ylabel="Multisite / one-site power",title="Shared-path diffraction")
    ax=axs[2]
    for r in data["planar"]:
        for s in r["results"]:
            a,b=s["indexes"][1],s["indexes"][-1]
            ax.scatter(s["quasi"][a]/s["quasi"][b],s["nominal_walkoff"][1]/s["nominal_walkoff"][-1],
                       s=12,color=BLUE,alpha=.6)
    lo,hi=ax.get_xlim(); ax.plot([lo,hi],[lo,hi],ls="--",lw=.75,color=GRAY)
    ax.set(xlabel="Group / weak-drive energy without walk-off",
           ylabel="Group / weak-drive energy with walk-off",title="Planar pulse model")
    ax=axs[3]
    errors=[]
    for r in data["planar"]:
        for s in r["results"]:
            if "integration_error" in s:
                errors.append([s["integration_error"],s["zero_walkoff_error"]])
    if errors:
        pairs=np.asarray(errors)
        ax.scatter(range(1,len(pairs)+1),pairs[:,0],color=BLUE,s=15,label="Refined time and propagation grids")
        ax.scatter(range(1,len(pairs)+1),pairs[:,1],color=ORANGE,s=15,label="Zero walk-off check")
    ax.set(yscale="log",xlabel="Checked geometry",xticks=range(1,len(errors)+1),
           ylabel="Absolute difference in pulse energy fraction",title="Planar solver checks")
    ax.legend()
    return fig


def supplement5(data):
    fig,axs=panels()
    rows=data["two_guides"]["rows"]
    for ax,(cohort,drive) in zip(axs,dict.fromkeys((r["cohort"],r["drive"]) for r in rows)):
        subset=[r for r in rows if (r["cohort"],r["drive"])==(cohort,drive)]
        base=[r["best_single"] for r in subset]
        counts=[x["counts"] for x in subset[0]["settings"]]
        pts=[ratio_ci([r["settings"][i]["selected_mean_power"] for r in subset],base) for i in range(len(counts))]
        interval_points(ax,counts,pts)
        ax.plot([],[],"o",ms=4,color=BLUE,label="12-probe selection")
        ax.axhline(1,ls="--",color=GRAY,lw=.7,label="Best single guide")
        ax.axhline(np.mean([r["equal_split"] for r in subset])/np.mean(base),ls=":",color=GREEN,lw=1,label="Equal split")
        ax.axhline(np.mean([r["oracle_power"] for r in subset])/np.mean(base),ls="-.",color=ORANGE,lw=.7,label="Dense phase sweep")
        ax.set(xscale="log",xticks=counts,xticklabels=counts,xlabel="Nominal counts per probe",
               ylabel="Power / best single guide",title=f'{cohort.replace("_"," ").capitalize()}, $g={drive:.2f}$')
    axs[0].legend()
    return fig


def save_figure(fig, directory, name):
    directory=Path(directory)
    directory.mkdir(parents=True,exist_ok=True)
    try:
        with (directory / f"{name}.pdf").open("wb") as pdf:
            fig.savefig(pdf, format="pdf")
        with (directory / f"{name}.png").open("wb") as png:
            fig.savefig(png, format="png", dpi=300)
        from IPython.display import Image, display

        display(Image(filename=str(directory / f"{name}.png"), width=760))
    finally:
        plt.close(fig)


In [ ]:
ARCHIVE_MEMBER = "refering data/nonlinear wavefront focusing/LN_single_spot.mat"
TENSOR_KEY = "TM_01_tensor"
RECEIVER_MODES = 36
GAUSSIAN_REPEATS = 1000
RMS_GRID = np.linspace(0.0, 1.0, 101)
SCALE_FACTORS = np.array([0.50, 0.75, 1.00, 1.25, 1.50])
RESPONSE = 0.69
OFFSET = 0.3415


def read_tensors(path):
    if path.suffix.lower() == ".zip":
        with ZipFile(path) as archive, TemporaryDirectory() as folder:
            mat_path = Path(folder) / "LN_single_spot.mat"
            with archive.open(ARCHIVE_MEMBER) as source, mat_path.open("wb") as target:
                copyfileobj(source, target)
            return read_tensors(mat_path)
    entries = loadmat(path, variable_names=[TENSOR_KEY])[TENSOR_KEY]
    if entries.shape != (RECEIVER_MODES * RECEIVER_MODES, 400) or not np.isfinite(entries).all():
        raise ValueError(f"Unexpected {TENSOR_KEY} entries or shape: {entries.shape}")
    tensors = entries.T.reshape(-1, RECEIVER_MODES, RECEIVER_MODES, order="F")
    residual = np.linalg.norm(tensors - tensors.transpose(0, 2, 1), axis=(1, 2))
    if np.any(residual > 1e-10 * np.linalg.norm(tensors, axis=(1, 2))):
        raise ValueError("The receiver tensors must be complex symmetric.")
    tensor_hash = hashlib.sha256(np.asarray(entries, dtype="<c16").tobytes()).hexdigest()
    return (tensors + tensors.transpose(0, 2, 1)) / 2, tensor_hash


def takagi_mask(matrix):
    lifted = np.block([[matrix.real, -matrix.imag], [-matrix.imag, -matrix.real]])
    singular, eigenvectors = eigh(lifted, subset_by_index=(2 * RECEIVER_MODES - 1, 2 * RECEIVER_MODES - 1))
    vector = eigenvectors[:RECEIVER_MODES, 0] + 1j * eigenvectors[RECEIVER_MODES:, 0]
    np.testing.assert_allclose(matrix @ vector, singular[0] * vector.conj(), atol=1e-10)
    return vector, np.exp(1j * np.angle(vector)) / np.sqrt(RECEIVER_MODES)


def phase_directions(matrix, vector, mask):
    local_field = mask * (matrix @ mask)
    receiver_field = local_field.sum()
    correction = -RESPONSE * RECEIVER_MODES * local_field.imag / (np.sqrt(RECEIVER_MODES) * abs(vector) + OFFSET)
    gradient = -4 * (receiver_field.conjugate() * local_field).imag
    return correction, unit_rms(gradient), -2 * local_field.imag


def rms(values):
    return np.sqrt(np.mean(values**2))


def unit_rms(values):
    norm = rms(values)
    return values / norm if norm > 0 else np.zeros_like(values)


def phase_scores(matrix, mask, phase_changes):
    waves = mask * np.exp(1j * phase_changes)
    fields = ((waves @ matrix) * waves).sum(axis=-1)
    return np.abs(fields) ** 2


def calibrate_gradient():
    rng = np.random.default_rng(532417)
    gains = []
    for _ in range(GAUSSIAN_REPEATS):
        gaussian = (rng.standard_normal((RECEIVER_MODES, RECEIVER_MODES)) + 1j * rng.standard_normal((RECEIVER_MODES, RECEIVER_MODES))) / np.sqrt(2)
        matrix = (gaussian + gaussian.T) / np.sqrt(2 * RECEIVER_MODES)
        matrix *= np.sqrt((2 / RECEIVER_MODES) / background(matrix))
        vector, mask = takagi_mask(matrix)
        _, gradient, _ = phase_directions(matrix, vector, mask)
        initial = phase_scores(matrix, mask, 0.0)
        gains.append(phase_scores(matrix, mask, RMS_GRID[:, None] * gradient) / initial)
    return float(RMS_GRID[np.argmax(np.mean(gains, axis=0))])


def receiver_search(matrix, seeds, background_value):
    def objective(phases):
        wave = np.exp(1j * phases) / np.sqrt(RECEIVER_MODES)
        local = wave * (matrix @ wave)
        return -float(local.sum().real), 2 * local.imag

    best_intensity = max(abs(seed.T @ matrix @ seed)**2 for seed in seeds)
    for seed in seeds:
        initial_field = seed.T @ matrix @ seed
        phases = np.angle(seed) - 0.5 * np.angle(initial_field)
        fit = minimize(objective, phases, jac=True, method="L-BFGS-B",
                       options={"maxiter": 250, "ftol": 1e-13, "gtol": 5e-9})
        wave = np.exp(1j * fit.x) / np.sqrt(RECEIVER_MODES)
        best_intensity = max(best_intensity, abs(wave.T @ matrix @ wave)**2)
    return best_intensity / background_value


def compare_receivers(tensors, fixed_rms):
    rng = np.random.default_rng(492118)
    records = []
    raw_background = np.array([background(tensor) for tensor in tensors])
    for receiver, (tensor, raw) in enumerate(zip(tensors, raw_background), start=1):
        denominator = 2 / RECEIVER_MODES
        matrix = tensor * np.sqrt(denominator / raw)
        vector, mask = takagi_mask(matrix)
        correction, gradient, field_gradient = phase_directions(matrix, vector, mask)
        initial = phase_scores(matrix, mask, 0.0) / denominator
        one_step = phase_scores(matrix, mask, correction) / denominator
        matched_rms = phase_scores(matrix, mask, rms(correction) * gradient) / denominator
        field_matched = phase_scores(matrix, mask, rms(correction) * unit_rms(field_gradient)) / denominator
        centered = field_gradient - field_gradient.mean()
        centered_rms = rms(correction - correction.mean())
        field_centered = phase_scores(matrix, mask, centered_rms * unit_rms(centered)) / denominator
        gradient_fixed = phase_scores(matrix, mask, fixed_rms * gradient) / denominator
        gradient_sweep = phase_scores(matrix, mask, RMS_GRID[:, None] * gradient) / denominator
        random_seeds = [np.exp(2j * np.pi * rng.random(RECEIVER_MODES)) / np.sqrt(RECEIVER_MODES) for _ in range(10)]
        direct = receiver_search(matrix, [mask, mask * np.exp(1j * correction), *random_seeds],
                                  denominator)
        record = {
            "receiver": receiver, "raw_background": float(raw), "phase_update_rms": float(rms(correction)),
            "projected_takagi": float(initial), "one_step_update": float(one_step),
            "field_gradient_matched_rms": float(field_matched),
            "field_gradient_centered_rms": float(field_centered),
            "intensity_gradient_matched_rms": float(matched_rms),
            "intensity_gradient_fixed_rms": float(gradient_fixed),
            "intensity_gradient_sweep": float(gradient_sweep.max()), "direct_search": float(direct),
            "gradient_sweep_best_rms": float(RMS_GRID[np.argmax(gradient_sweep)]),
        }
        for factor in SCALE_FACTORS:
            record[f"one_step_scale_{factor:.2f}"] = float(
                phase_scores(matrix, mask, factor * correction) / denominator)
        global_scale = np.sqrt(raw / np.median(raw_background))
        record["one_step_common_scale"] = float(
            phase_scores(matrix, mask, global_scale * correction) / denominator)
        records.append(record)
    return records


def summarize_receivers(records, fixed_rms, tensor_hash):
    initial = np.array([row["projected_takagi"] for row in records])
    one_step = np.array([row["one_step_update"] for row in records])
    methods = [
        "projected_takagi", "one_step_update", "field_gradient_matched_rms",
        "field_gradient_centered_rms", "intensity_gradient_matched_rms",
        "intensity_gradient_fixed_rms", "intensity_gradient_sweep", "direct_search",
        *(f"one_step_scale_{factor:.2f}" for factor in SCALE_FACTORS), "one_step_common_scale",
    ]
    summary = {}
    for method in methods:
        values = np.array([row[method] for row in records])
        gains = 100 * (values / initial - 1)
        summary[method] = {
            "mean_enhancement": float(values.mean()),
            "mean_paired_gain_percent": float(gains.mean()),
            "gain_quantiles_percent": dict(zip(
                ["minimum", "q25", "median", "q75", "maximum"],
                np.quantile(gains, [0, 0.25, 0.50, 0.75, 1]).tolist())),
            "number_better_than_takagi": int(np.sum(values > initial + 1e-9)),
            "number_worse_than_takagi": int(np.sum(values < initial - 1e-9)),
        }
        if "_gradient_" in method:
            difference = 100 * (one_step - values) / initial
            summary[method]["median_update_minus_gradient_percentage_points"] = float(np.median(difference))
            summary[method]["number_update_outperforms_gradient"] = int(np.sum(one_step > values + 1e-9))
            summary[method]["number_gradient_outperforms_update"] = int(np.sum(values > one_step + 1e-9))
    return {
        "data_doi": "10.5281/zenodo.19246102", "mat_member": ARCHIVE_MEMBER,
        "tensor_variable": TENSOR_KEY, "tensor_entries_sha256": tensor_hash,
        "modes": RECEIVER_MODES, "receivers": len(records), "gradient_calibration_rms_radians": fixed_rms,
        "gradient_calibration_gaussian_tensors": GAUSSIAN_REPEATS,
        "gradient_sweep_boundary_count": int(sum(row["gradient_sweep_best_rms"] == RMS_GRID[-1]
                                                 for row in records)),
        "search_gap_recovered_fraction": float(np.sum(one_step - initial) / np.sum(
            np.array([row["direct_search"] for row in records]) - initial)),
        "methods": summary,
    }


def receiver_gain(records, key):
    initial = np.array([row["projected_takagi"] for row in records])
    values = np.array([row[key] for row in records])
    return 100 * (values / initial - 1)


def count_receiver_wins(records, key):
    return sum(row["one_step_update"] > row[key] + 1e-9 for row in records)


def calculate_optical(path):
    tensors, tensor_hash = read_tensors(path)
    fixed_rms = calibrate_gradient()
    records = compare_receivers(tensors, fixed_rms)
    return records, summarize_receivers(records, fixed_rms, tensor_hash)


def write_optical_results(directory, records, summary):
    with (directory / "receiver_gains.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
    (directory / "receiver_summary.json").write_text(
        json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )


In [ ]:
style()
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
outputs = Path(mkdtemp(prefix="run-", dir=RESULTS_DIRECTORY.resolve()))
print(f"Writing results to {outputs}", flush=True)

records, optical_summary = calculate_optical(DATA_PATH)
write_optical_results(outputs, records, optical_summary)

data = calculate_simulations()
(outputs / "simulation_results.json").write_text(
    json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
data["optical"] = records


In [ ]:
save_figure(figure1(data), outputs, "Figure_1")


In [ ]:
save_figure(figure2(data), outputs, "Figure_2")


In [ ]:
save_figure(figure3(data), outputs, "Figure_3")


In [ ]:
save_figure(figure4(data), outputs, "Figure_4")


In [ ]:
save_figure(figure5(data), outputs, "Figure_5")


In [ ]:
for i, make_figure in enumerate(
    (supplement1, supplement2, supplement3, supplement4, supplement5), 1
):
    save_figure(make_figure(data), outputs, f"Supplementary_Figure_{i}")

print(f"Saved figures to {outputs}")


In [ ]:
summary = {
    "gaussian": {
        "asymptotic_lower": data["dense_reference"]["analytic_lower"]["asymptotic_enhancement_per_mode"],
        "asymptotic_upper": data["dense_reference"]["analytic_upper"]["continuous_enhancement_per_mode"],
        "at_1024": data["dense_reference"]["rows"][-1]["summary"]["feasible"],
    },
    "half_intensity_crossings_at_192": {
        row["distribution"]: {"value": row["half"], "ci95": row["bootstrap_half_ci"]}
        for row in data["crossover"] if row["n"] == 192
    },
    "passive_small_mean_percent_of_launch": {
        key: float(100 * np.mean([row[key] for row in data["passive_small"]]))
        for key in ("one", "weak_mask_at_drive", "simple_multi", "optimized")
    },
}
baseline = [row for row in data["coupling"] if (row["hopping"], row["disorder"]) == (.45, .45)]
summary["moderate_coupling_group_to_weak_reference"] = ratio_ci(
    [row["best_group"] for row in baseline], [row["weak"] for row in baseline]
).tolist()

diffraction = [by_drive(row, 16) for row in data["diffraction"]]
groups = [max(power for size, power in zip(row["counts"], row["groups"]) if size >= 2)
          for row in diffraction]
summary["diffraction_group_to_one_site"] = ratio_ci(groups, [row["single"] for row in diffraction]).tolist()
summary["diffraction_group_to_weak_reference"] = ratio_ci(
    groups, [max(row["weak"], row["old_weak"]) for row in diffraction]
).tolist()
summary["planar_group_to_weak_reference"] = []
for drive_index in (0, 1, -1):
    energies = [row["results"][drive_index]["nominal_walkoff"] for row in data["planar"]]
    summary["planar_group_to_weak_reference"].append(
        ratio_ci([row[1] for row in energies], [row[-1] for row in energies]).tolist()
    )

summary["receiver_count"] = len(records)
summary["optical_median_gain_percent"] = {
    method: optical_summary["methods"][method]["gain_quantiles_percent"]["median"]
    for method in ("one_step_update", "field_gradient_matched_rms", "intensity_gradient_sweep")
}
summary["analytic_update_wins"] = {
    method: count_receiver_wins(records, method)
    for method in ("field_gradient_matched_rms", "intensity_gradient_sweep")
}
(outputs / "paper_values.json").write_text(
    json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Figures and source data saved in {outputs}")
